In [ ]:
import numpy as np
import pandas as pd
import os, sys, glob
import matplotlib.pyplot as plt
import seaborn as sns
import nibabel as nib
import scipy
import scipy.stats as stats
from scipy.spatial.distance import cdist, pdist, squareform
from scipy.stats import ttest_rel, ttest_ind, wilcoxon,spearmanr
import nilearn
from nilearn import plotting, image, datasets
from nilearn.maskers import NiftiMasker
from nilearn.mass_univariate import permuted_ols
from matplotlib.colors import ListedColormap
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.gridspec as gridspec
from patsy import dmatrix
from sklearn.utils import resample
import pingouin as pg
from obspy.imaging.cm import viridis_white, viridis_white_r
import plotting_helpers as helper
import stats_helpers as sh
import stats_helpers as stats_helpers
import parcelwise_regressions as pwr
import parcelwise_regressions as pr
import narratives_utils as naru
import narratives_config as narc
import adult_restmovie_utils as aru
import adult_restmovie_config as arc
import infant_restmovie_utils as iru
import infant_restmovie_config as irc
import hbn_utils as hu
import hbn_config as hc
import partlycloudy_utils as pcu
import partlycloudy_config as pcc
from matplotlib import rcParams
from scipy.stats import spearmanr
from scipy.spatial.distance import squareform

rcParams['pdf.fonttype'] = 42
rcParams['ps.fonttype'] = 42
%load_ext autoreload
%autoreload 2

PLOT_DIR = 'compiled/plots'
os.makedirs(PLOT_DIR, exist_ok=True)
ISC_CMAP='OrRd'
ID_CMAP=helper.custom_blues_cmap()
import matplotlib.ticker as mticker
from scipy.stats import gaussian_kde
dataset_colors = helper.dataset_colors('all')

In [ ]:
# Common functions
def expand_parcellation_to_volume(data_arr, atlas_img):
    atlas_data = atlas_img.get_fdata()
    vol_data = np.zeros_like(atlas_data)
    for i, val in enumerate(data_arr):
        if not np.isnan(val):
            vol_data[atlas_data == (i + 1)] = val
    return nib.Nifti1Image(vol_data, atlas_img.affine, atlas_img.header)

def get_average_results(df, col_names=[], target_names=[], value_name='score', region_name_order=None):
    # Filter the DataFrame to include only the specified columns and target names
    for i in range(len(col_names)):
        df = df[df[col_names[i]] == target_names[i]].reset_index(drop=True)
    # Average by region name
    filtered_df = df.groupby('region_name').mean(numeric_only=True).reset_index()
    # Reorder
    if region_name_order is not None:
        filtered_df = reorder_region_names(filtered_df, region_name_order)
    return filtered_df[value_name].values

def reorder_region_names(df, region_name_order):
    df['region_name'] = pd.Categorical(df['region_name'], categories=region_name_order, ordered=True)
    return df.sort_values('region_name').reset_index(drop=True)

def load_atlas():
    ATLAS = datasets.fetch_atlas_schaefer_2018(resolution_mm=2, n_rois=400, yeo_networks=17)
    ATLAS.labels = np.insert(ATLAS.labels, 0, 'Background')
    atlas_img = nib.load(ATLAS.maps)
    labels = [a.decode('utf-8') for a in ATLAS.labels[1:]]
    return atlas_img, labels

def get_region_order():
    _, labels = load_atlas()
    return labels

def lme_fit_summary(model):
    """Comprehensive fit summary for LME model"""
    
    # R² calculations
    y = model.model.endog
    fitted = model.fittedvalues
    
    var_fixed = np.var(fitted)
    var_random = model.cov_re.iloc[0, 0]
    var_residual = model.scale
    var_total = var_fixed + var_random + var_residual
    
    r2_marginal = var_fixed / var_total
    r2_conditional = (var_fixed + var_random) / var_total
    
    # Correlation-based
    r2_corr = stats.pearsonr(y, fitted)[0] ** 2
    
    # ICC (Intraclass Correlation)
    icc = var_random / (var_random + var_residual)
    
    print("=" * 70)
    print("LINEAR MIXED EFFECTS MODEL FIT SUMMARY")
    print("=" * 70)
    print(f"\nR² Statistics:")
    print(f"  Marginal R² (fixed only):       {r2_marginal:.3f}")
    print(f"  Conditional R² (fixed+random):  {r2_conditional:.3f}")
    print(f"  Simple R² (correlation):        {r2_corr:.3f}")
    print(f"\nVariance Decomposition:")
    print(f"  Fixed effects:      {var_fixed:>10.4f}  ({100*var_fixed/var_total:>5.1f}%)")
    print(f"  Random effects:     {var_random:>10.4f}  ({100*var_random/var_total:>5.1f}%)")
    print(f"  Residual:           {var_residual:>10.4f}  ({100*var_residual/var_total:>5.1f}%)")
    print(f"  Total:              {var_total:>10.4f}")
    print(f"\nIntraclass Correlation (ICC):     {icc:.3f}")
    print(f"  (Proportion of variance due to clustering)")
    print(f"\nModel Comparison:")
    print(f"  AIC:                {model.aic:.1f}")
    print(f"  BIC:                {model.bic:.1f}")
    print(f"  Log-Likelihood:     {model.llf:.1f}")
    print(f"\nSample:")
    print(f"  N observations:     {len(y)}")
    print(f"  N groups:           {len(np.unique(model.model.groups))}")
    print("=" * 70)
    
    return {
        'r2_marginal': r2_marginal,
        'r2_conditional': r2_conditional,
        'r2_correlation': r2_corr,
        'icc': icc,
        'aic': model.aic,
        'bic': model.bic
    }



# Basics: age and n distributions

In [ ]:
# Participant ages
par_df = pd.read_csv(f'compiled/info/combined_participant_info.csv', index_col=0)

In [ ]:
# Group participants from PartlyCloudy into 5 bins: <5, 5-6, 6-8.5, 8.5-13, Adults
pc_pars = par_df[par_df['dataset'] == 'PartlyCloudy']
pc_pars['age_years'] = pc_pars.age_months / 12
partlycloudy_bins = [0, 5, 6, 8.5, 13, np.inf]
partlycloudy_labels = ['<5', '5-6', '6-8.5', '8.5-13', 'Adults']
pc_pars['age_bin'] = pd.cut(pc_pars['age_years'], bins=partlycloudy_bins, labels=partlycloudy_labels)
pc_pars.groupby('age_bin').count()

In [ ]:
hb_pars = par_df[par_df['dataset']=='HBN']
hb_pars['age_years'] = hb_pars.age_months / 12
bins = [0, 10, 13, 17, np.inf]
labels = ['<10', '10-13', '13-17', '>17']
hb_pars['age_bin'] = pd.cut(hb_pars['age_years'], bins=bins, labels=labels)
hb_pars.groupby('age_bin').count()

In [ ]:
par_df[par_df['dataset']=='InfantRestMovie']['age_months'].describe()

In [ ]:
# ── load & prep ───────────────────────────────────────────────────────────────
_info = pd.read_csv('compiled/info/combined_participant_info.csv')

_name_map = {
    'AdultRestMovie':  'adult_restmovie',
    'HBN':             'hbn',
    'InfantRestMovie': 'infant_restmovie',
    'Narratives':      'narratives',
    'PartlyCloudy':    'partlycloudy',
}
_label_map = {
    'infant_restmovie': 'Infant Rest/Movie',
    'partlycloudy':     'Partly Cloudy',
    'hbn':              'Healthy Brain Network',
    'adult_restmovie':  'Adult Rest/Movie',
    'narratives':       'Narratives',
}

_info['dataset_key'] = _info['dataset'].map(_name_map)
_info['age_years']   = _info['age_months'] / 12.0

_subs = (
    _info.drop_duplicates(subset=['participant_id', 'dataset_key'])
         [['participant_id', 'dataset_key', 'age_years']]
)

# oldest-to-youngest order
_order = (
    _subs.groupby('dataset_key')['age_years']
         .median().sort_values().index.tolist()
)[::-1] #

# ── x-axis tick positions / labels ───────────────────────────────────────────
# every 3 months up to 24 months, then every 5 years
_mo_vals   = np.arange(0, 30, 4) / 12.0          # 0, 3, 6 … 24 months in years
_yr_vals   = np.arange(5, 56, 5, dtype=float)     # 5, 10 … 55 years
_all_ticks = np.concatenate([_mo_vals, _yr_vals])

def _fmt(t):
    m = round(t * 12)

    if m == 0:  return '0'
    if m < 25:  return f''
    return f'{int(round(t))}'

_tick_labels = [_fmt(t) for t in _all_ticks]

_x0  = 0.0
_x1  = _subs['age_years'].max() + 1
_xgr = np.linspace(_x0, _x1, 1200)

# ── figure ────────────────────────────────────────────────────────────────────
n_ds  = len(_order)
fig, axes = plt.subplots(n_ds, 1, figsize=(8, n_ds * 1.5),
                          sharex=True, facecolor='white')
fig.subplots_adjust(hspace=0.35)

for i, (key, ax) in enumerate(zip(_order, axes)):
    ages    = _subs.loc[_subs['dataset_key'] == key, 'age_years'].dropna().values
    n_sub   = len(ages)
    lo, hi  = ages.min(), ages.max()
    color   = dataset_colors[key]

    kde   = gaussian_kde(ages, bw_method='scott')
    ykde  = kde(_xgr)
    ynorm = ykde / ykde.max()   # normalise to [0, 1]

    # filled ridge with black outline
    ax.fill_between(_xgr, 0, ynorm, color=color, alpha=0.80, linewidth=0, zorder=2)
    ax.plot(_xgr, ynorm, color='black', linewidth=1.2, zorder=3)
    ax.axhline(0, color='black', linewidth=0.8, zorder=1)

    # ── y-axis ────────────────────────────────────────────────────────────────
    ax.set_ylim(-0.05, 1.2)
    ax.set_yticks([0.0, 0.5, 1.0])
    ax.set_yticklabels(['0', '0.5', '1'], fontsize=8, color='black')
    ax.spines['left'].set_color('black')
    ax.spines['left'].set_linewidth(1)

    # ── x-axis (shown on last panel) ─────────────────────────────────────────
    if i == n_ds - 1:
        ax.spines['bottom'].set_color('black')
        ax.spines['bottom'].set_linewidth(1.0)
        ax.tick_params(axis='x', which='both', bottom=True, labelbottom=True,
                    colors='black', direction='out', length=4, labelsize=8)
    else:
        ax.spines['bottom'].set_visible(False)
        ax.tick_params(axis='x', which='both', bottom=False, labelbottom=False)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # ── right-justified label ─────────────────────────────────────────────────
    ax.text(
        0.9, .95,
        _label_map[key],
        transform=ax.transAxes,
        ha='right', va='top',
        fontsize=16, fontweight='bold', color=color, zorder=5,
    )
    ax.text(
        0.9, .74,
        f'N={n_sub},   {lo:.1f}\u2013{hi:.1f} yrs',
        transform=ax.transAxes,
        ha='right', va='top',
        fontsize=12, color='#333333', zorder=5,
    )

# ── shared x-axis ticks (applied once, propagate via sharex) ─────────────────
axes[-1].set_xticks(_all_ticks)

axes[-1].set_xticklabels(_tick_labels, fontsize=12)
axes[-1].set_xlim(_x0, _x1)

# ── shared y-label ────────────────────────────────────────────────────────────
fig.text(0.01, 0.5, 'Normalised density', va='center', ha='center',
         rotation='vertical', fontsize=12, color='black')

axes[0].set_title('Dataset age distributions', fontsize=16, pad=16)
fig.tight_layout()
plt.savefig('compiled/plots/age_distributions_ridge.pdf', bbox_inches='tight',
            dpi=300, transparent=True)
plt.show()

# Section 1: Narratives

## 1.1 Surface grid: ISC and IDE by task

In [ ]:
# Narratives: load results directory, define tasks/measures, load atlas
narrative_tasks = ['black', 'bronx', 'forgot', 'piemanpni']
measures = {'ISC': 'ISC', 'TPHATE_DiffOp_IDE': 'IDE'}
atlas_img, atlas_labels = load_atlas()

# Load csv file
results_df_nar = pd.read_csv(f'compiled/results/narratives_compiled_results.csv')
avg_results = {}
for task in narrative_tasks:
    avg_results[task] = {}
    for measure in measures.keys():
        avg_results[task][measure] = get_average_results(results_df_nar, col_names=['task', 'measure'], target_names=[task, measure], value_name='score', region_name_order=atlas_labels)
        print(f"Loaded {task} {measure}: shape {avg_results[task][measure].shape}, mean={np.nanmean(avg_results[task][measure]):.3f}")

# Narratives: surface grid (4 tasks x 2 measures)
nifti_images, titles, cbar_ranges, cmaps = [], [], [], []

for measure, label in measures.items():
    for task in narrative_tasks:
        if measure in avg_results[task]:
            vol_img = expand_parcellation_to_volume(avg_results[task][measure], atlas_img)
            nifti_images.append(vol_img)
            titles.append(f'{task.capitalize()}')
            if measure == 'ISC':
                cbar_ranges.append((0, 0.4)); cmaps.append(ISC_CMAP)
            else:
                cbar_ranges.append((1, 32)); cmaps.append(ID_CMAP)


temp_fns = []
for idx, (img, title, cmap, cbar_range) in enumerate(zip(nifti_images, titles, cmaps, cbar_ranges)):
    temp_fn = f'/tmp/narratives_plot_{idx}.png'
    helper.generate_surface_plot(
        data_fn=img, image_fn=temp_fn, atlas='searchlight', cmap=cmap,
        cbar_range=cbar_range, surf_type='fslr', target_density='32k',
        include_cbar=False, title=title, method='linear', threshold=None, mask_medial_wall=True
    )
    temp_fns.append(temp_fn)

helper.compile_surface_plots_to_grid_by_rows(
    image_files=temp_fns, n_rows=2, atlas='searchlight', data_files=nifti_images,
    surf_type='fslr', target_density='32k',
    output_path=f'{PLOT_DIR}/narratives_all_tasks_isc_ide_surface_grid.pdf',
    main_title='Narratives dataset',
    cmaps_per_row=[ISC_CMAP, cmap],
    cbar_ranges_per_row=[[0, 0.4], [1, 34]],
    cbar_labels_per_row=["Pearson's r", 'Dimensionality']
)


## 1.2 Cross-task reliability (ISC vs IDE Spearman heatmap)

In [ ]:
# Narratives: cross-task Spearman reliability heatmap (Bonferroni corrected)


temp = results_df_nar.groupby(['region_name', 'task', 'measure']).mean(numeric_only=True).reset_index()
isc_piv = temp[temp['measure'] == 'ISC'].pivot_table(index='region_name', columns='task', values='score').values
ide_piv = temp[temp['measure'] == 'TPHATE_DiffOp_IDE'].pivot_table(index='region_name', columns='task', values='score').values
corrmat, pvals = spearmanr(isc_piv, ide_piv)

ind = np.triu_indices_from(pvals, k=0)
pvals_lower = pvals.copy()
pvals_lower[ind] = np.nan
corrmat_lower = corrmat.copy()
corrmat_lower[ind] = np.nan
np.fill_diagonal(corrmat_lower, 1)

reject, _, _, _ = sh.multipletests(pvals_lower[pvals_lower == pvals_lower], method='bonferroni', alpha=0.05)
reject_mat = squareform(reject)
# set diagonal to True since we want to show the diagonal values
np.fill_diagonal(reject_mat, True)

fig, ax = plt.subplots(figsize=(7, 6))
g = sns.heatmap(corrmat_lower * reject_mat, annot=True, fmt=".2f",
                cmap=helper.diverging_colormap_gp(), linecolor='k', linewidths=0.5,
                vmin=-1, vmax=1, square=True, xticklabels=False, yticklabels=False, ax=ax)
cbar = g.collections[0].colorbar
cbar.set_ticks([-1, -0.5, 0, 0.5, 1])
cbar.outline.set_edgecolor('black')
cbar.outline.set_linewidth(1)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/narratives_cross_task_reliability_heatmap.pdf', format='pdf', transparent=True)
plt.show()

## 1.3 Average ISC, IDE, and ISC-IDE correlation by task

In [ ]:
# Narratives: load ISC-IDE correlation CSV
df_corr_nar = pd.read_csv('compiled/results/combined_corr_analyses.csv',index_col=0)
df_corr_nar = df_corr_nar[df_corr_nar['dataset'] == 'narratives'].reset_index()
t, p = stats.ttest_1samp(df_corr_nar['zscore'].values, 0, alternative='two-sided')
print(f"t-statistic: {t:.3f}, p-value: {p:.3e}, zscore mean: {df_corr_nar['zscore'].mean():.3f}, degrees of freedom: {len(df_corr_nar)-1}")

In [ ]:
# ISC-IDE z-score barplot
tasks = ['black', 'bronx', 'forgot', 'piemanpni']
corr_color = helper.dataset_colors('narratives')

fig,ax=plt.subplots(figsize=(5, 4))
g = sns.barplot(x='task', y='zscore', data=df_corr_nar,
                ax=ax, color=corr_color, edgecolor='k', linewidth=1, alpha=0.6)
sns.stripplot(x='task', y='zscore', data=df_corr_nar,
              ax=ax, color=corr_color, size=6, edgecolor='k', linewidth=0.5, jitter=False, alpha=1)
for sub in df_corr_nar['subject'].unique():
    sub_data = df_corr_nar[df_corr_nar['subject'] == sub].sort_values('task')
    if len(sub_data) == 4:
        ax.plot(range(4), sub_data['zscore'].values, color='gray', alpha=1, linewidth=0.2, zorder=1)
for i, task in enumerate(tasks):
    task_data = df_corr_nar[df_corr_nar['task'] == task]
    t, p = stats.ttest_1samp(task_data['zscore'], 0, alternative='two-sided')
    print(f"t-statistic: {t:.3f}, p-value: {p:.3e}")
    ax.text(i, 11, helper.get_asterisks_pvalue(p), ha='center', va='bottom', fontsize=14)
ax.axhline(0, color='k', linestyle='--', linewidth=1)
g.set(xlabel='', ylabel='Z-score', title='ISC-ID relationship', 
      ylim=(-6, 13), xticklabels=['Black', 'Bronx', 'Forgot', 'Pieman'], yticks=range(-6, 13, 3))
sns.despine()
# plt.savefig(f'{PLOT_DIR}/narratives_isc_ide_barplot.pdf', format='pdf', transparent=True)


In [ ]:
# Narratives: three barplots — mean ISC, mean IDE, ISC-IDE z-score by task
tasks = ['black', 'bronx', 'forgot', 'piemanpni']
ISC_Color = helper.dataset_colors('narratives') #sns.color_palette("magma")[1]
IDE_Color = helper.dataset_colors('narratives') #sns.color_palette('viridis')[1]
corr_color = helper.dataset_colors('narratives')

temp = results_df_nar.groupby(['task', 'measure', 'subject_id']).mean(numeric_only=True).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(10, 4))

# ISC barplot
g = sns.barplot(x='task', y='score', data=temp[temp['measure'] == 'ISC'],
                ax=axes[0], color=ISC_Color, edgecolor='k', linewidth=1, alpha=0.6)
sns.stripplot(x='task', y='score', data=temp[temp['measure'] == 'ISC'],
              ax=axes[0], color=ISC_Color, size=6, edgecolor='k', linewidth=0.5, jitter=False, alpha=1)
for sub in temp['subject_id'].unique():
    sub_data = temp[(temp['subject_id'] == sub) & (temp['measure'] == 'ISC')].sort_values('task')
    if len(sub_data) == 4:
        axes[0].plot(range(4), sub_data['score'].values, color='gray', alpha=1, linewidth=0.2, zorder=1)
g.set(xlabel='Task', ylabel="Pearson's r", title='Whole-brain average ISC',  xticklabels=['Black', 'Bronx', 'Forgot', 'Pieman'], 
      yticks=np.arange(0,0.13,0.03))

# IDE barplot
g = sns.barplot(x='task', y='score', data=temp[temp['measure'] == 'TPHATE_DiffOp_IDE'],
                ax=axes[1], color=IDE_Color, edgecolor='k', linewidth=1, alpha=0.6)
sns.stripplot(x='task', y='score', data=temp[temp['measure'] == 'TPHATE_DiffOp_IDE'],
              ax=axes[1], color=IDE_Color, size=6, edgecolor='k', linewidth=0.5, jitter=False, alpha=1)
for sub in temp['subject_id'].unique():
    sub_data = temp[(temp['subject_id'] == sub) & (temp['measure'] == 'TPHATE_DiffOp_IDE')].sort_values('task')
    if len(sub_data) == 4:
        axes[1].plot(range(4), sub_data['score'].values, color='gray', alpha=1, linewidth=0.2, zorder=1)
g.set(xlabel='Task', ylabel='# Dimensions', title='Whole-brain average IDE',  xticklabels=['Black', 'Bronx', 'Forgot', 'Pieman'], yticks=np.arange(0,22,5))

# ISC-IDE z-score barplot
g = sns.barplot(x='task', y='zscore', data=df_corr_nar,
                ax=axes[2], color=corr_color, edgecolor='k', linewidth=1, alpha=0.6)
sns.stripplot(x='task', y='zscore', data=df_corr_nar,
              ax=axes[2], color=corr_color, size=6, edgecolor='k', linewidth=0.5, jitter=False, alpha=1)
for sub in df_corr_nar['subject'].unique():
    sub_data = df_corr_nar[df_corr_nar['subject'] == sub].sort_values('task')
    if len(sub_data) == 4:
        axes[2].plot(range(4), sub_data['zscore'].values, color='gray', alpha=1, linewidth=0.2, zorder=1)
for i, task in enumerate(tasks):
    task_data = df_corr_nar[df_corr_nar['task'] == task]
    t, p = stats.ttest_1samp(task_data['zscore'], 0, alternative='two-sided')
    axes[2].text(i, 11, helper.get_asterisks_pvalue(p), ha='center', va='bottom', fontsize=14)
axes[2].axhline(0, color='k', linestyle='--', linewidth=1)
g.set(xlabel='', ylabel='Z-score', title='ISC-ID relationship', ylim=(-6, 12), xticklabels=['Black', 'Bronx', 'Forgot', 'Pieman'], yticks=range(-6, 13, 3))

sns.despine()
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/narratives_avg_isc_ide_corr_barplots.pdf', format='pdf', transparent=True)
plt.show()

# Section 2: Adult Rest/Movie

In [ ]:
ttest_ind(adult_zscores, infant_zscores, equal_var=False)

In [ ]:
corrs = pd.read_csv('compiled/results/combined_corr_analyses.csv', index_col=0)
corrs = corrs[corrs['dataset'].isin(['infant_restmovie','adult_restmovie'])].reset_index()
# Plot a bar plot comparing the ISC-IDE correlation z-scores between infants and adults
fig, ax = plt.subplots(figsize=(3,4))
sns.barplot(x='dataset', y='zscore', data=corrs, order=['infant_restmovie','adult_restmovie'], ax=ax, palette=[helper.dataset_colors('infant_restmovie'), helper.dataset_colors('adult_restmovie')], edgecolor='k', linewidth=1, alpha=0.6)
sns.stripplot(x='dataset', y='zscore', data=corrs,jitter=True, order=['infant_restmovie','adult_restmovie'], ax=ax, palette=[helper.dataset_colors('infant_restmovie'), helper.dataset_colors('adult_restmovie')], 
              size=6, edgecolor='k', linewidth=0.5,  alpha=1)
for sub in corrs['subject'].unique():
    sub_data = corrs[corrs['subject'] == sub].sort_values('dataset')
    if len(sub_data) == 2:
        ax.plot([0, 1], sub_data['zscore'].values, color='gray', alpha=1, linewidth=0.5, zorder=1)
# compare the ISC-IDE correlation z-scores between infants and adults using a t-test
infant_zscores = corrs[corrs['dataset'] == 'infant_restmovie']['zscore']
adult_zscores = corrs[corrs['dataset'] == 'adult_restmovie']['zscore']
t_stat, p_value = ttest_ind(infant_zscores, adult_zscores, equal_var=False)
print(f'T-test results: t-statistic = {t_stat:.3f}, p-value = {p_value:.3f}')
# Also compare each to 0
t_stat_inf, p_value_inf = ttest_rel(infant_zscores, np.zeros_like(infant_zscores))
t_stat_adult, p_value_adult = ttest_rel(adult_zscores,  np.zeros_like(adult_zscores))
mi = np.mean(infant_zscores)
ma = np.mean(adult_zscores)
print(f'Infant group vs 0: mean={mi:.3f}, t-statistic = {t_stat_inf:.3f}, p-value = {p_value_inf:.3f}')
print(f'Adult group vs 0: mean={ma:.3f},  t-statistic = {t_stat_adult:.3f}, p-value = {p_value_adult:.3f}')
# Add significance asterisks
if p_value < 0.05:
    ax.text(0.5, 17, helper.get_asterisks_pvalue(p_value), ha='center', va='bottom', fontsize=14)

ax.set_ylim=(-6, 18)

ax.axhline(0, color='k', linestyle='--', linewidth=1)
ax.set_xlabel('')
ax.set_ylabel('z-score')
ax.set_title('ISC-ID relationship', fontsize=12)
sns.despine()
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/isc_id_corr_zscore_comparison_infants_adults.pdf', format='pdf', transparent=True)
plt.show()

In [ ]:
# Adult Rest/Movie: setup and load average ISC/IDE volumes for aeronaut
results_dir_arm = './adult_restmovie/results'
movie_tasks_arm = ['aeronaut','mickey','rest']
measures_arm = {'ISC': 'ISC', 'TPHATE_DiffOp_IDE': 'IDE'}
avg_results_arm = {}
for task in movie_tasks_arm:
    avg_results_arm[task] = {}
    for measure in measures_arm.keys():
        fn = f'{results_dir_arm}/{task}_{measure}_all_subjects_results.nii.gz'
        if os.path.exists(fn):
            nii = nib.load(fn)
            # Average across the subject dimension (assuming it's the last dimension)
            avg = image.mean_img(nii)
            print(f"Loaded {task} {measure}: shape {avg.shape}")
            avg_results_arm[task][measure] = avg
        else:
            print(f"Warning: {fn} not found")

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

helper.generate_surface_plot(
    data_fn=avg_results_arm['aeronaut']['TPHATE_DiffOp_IDE'],
    image_fn=f'{PLOT_DIR}/adult_restmovie_aeronaut_ide_surface.pdf',
    atlas='searchlight', cmap=ID_CMAP, cbar_range=(1, 18),
    surf_type='fslr', target_density='32k', include_cbar=True,
    title='Aeronaut IDE', method='linear', threshold=None, mask_medial_wall=True
)

helper.generate_surface_plot(
    data_fn=avg_results_arm['aeronaut']['ISC'],
    image_fn=f'{PLOT_DIR}/adult_restmovie_aeronaut_isc_surface.pdf',
    atlas='searchlight', cmap=ISC_CMAP, cbar_range=(0, 0.4),
    surf_type='fslr', target_density='32k', include_cbar=True,
    title='Aeronaut ISC', method='linear', threshold=None, mask_medial_wall=True
)

helper.generate_surface_plot(
    data_fn=avg_results_arm['mickey']['TPHATE_DiffOp_IDE'],
    image_fn=f'{PLOT_DIR}/adult_restmovie_mickey_ide_surface.pdf',
    atlas='searchlight', cmap=ID_CMAP, cbar_range=(1, 18),
    surf_type='fslr', target_density='32k', include_cbar=True,
    title='Mickey IDE', method='linear', threshold=None, mask_medial_wall=True
)


helper.generate_surface_plot(
    data_fn=avg_results_arm['rest']['TPHATE_DiffOp_IDE'],
    image_fn=f'{PLOT_DIR}/adult_restmovie_rest_ide_surface.pdf',
    atlas='searchlight', cmap=ID_CMAP, cbar_range=(1, 25),
    surf_type='fslr', target_density='32k', include_cbar=True,
    title='Rest IDE', method='linear', threshold=None, mask_medial_wall=True
)



In [ ]:
# Adult Rest/Movie: load aeronaut-rest difference maps
diff_maps_dir = f'{results_dir_arm}/'
diff_results_arm = {}
for pair in ['aeronaut_rest']:
    diff_results_arm[pair] = {}
    for kind, suffix in [('thresholded', 'thresholded'), ('unthresholded', 'unthresholded')]:
        fn = f'{diff_maps_dir}/{pair}_TPHATE_DiffOp_IDE_SL_difference_maps_avg_{suffix}.nii.gz'
        if os.path.exists(fn):
            diff_results_arm[pair][kind] = nib.load(fn)
            helper.generate_surface_plot(
                data_fn=diff_results_arm[pair][kind],
                image_fn=f'{PLOT_DIR}/adult_restmovie_rest_aeronaut_surface.pdf',
                atlas='searchlight', cmap=helper.diverging_colormap_bp(), cbar_range=(-9, 9),
                surf_type='fslr', target_density='32k', include_cbar=True,
                title='Rest IDE', method='linear', threshold=None, mask_medial_wall=True
                )
            print(f"Loaded {pair} {kind}")
        else:
            print(f"Warning: {fn} not found")


In [ ]:
# Adult Rest/Movie: mean IDE barplot by task (aeronaut, mickey, rest)
ide_subject_averages_arm = []
mask = nib.load(f'adult_restmovie/masks/adult_restmovie_intersect_mask.nii.gz')
masker = NiftiMasker(mask_img=mask)
for task in ['aeronaut', 'rest']:
    ide_fn = f'{results_dir_arm}/{task}_TPHATE_DiffOp_IDE_all_subjects_results.nii.gz'
    if os.path.exists(ide_fn):
        ide_img = nib.load(ide_fn)
        ide_data = masker.fit_transform(ide_img)
        n_subjects = ide_data.shape[0]
        for subj_idx in range(n_subjects):
            mean_ide = np.nanmean(ide_data[subj_idx])
            ide_subject_averages_arm.append({'task': task, 'subject_id': subj_idx, 'mean_IDE': mean_ide, "std_IDE": np.nanstd(ide_data[subj_idx])})
        print(f"Loaded {task}: {n_subjects} subjects, mean IDE across subjects = {np.nanmean(ide_data):.3f}, sd = {np.mean(np.nanstd(ide_data, axis=1)):.3f}")
    else:
        print(f"Warning: {ide_fn} not found"), 

ide_df_arm = pd.DataFrame(ide_subject_averages_arm)
cmap_arm = [helper.dataset_colors('adult_restmovie'), helper.dataset_colors('adult_restmovie'), helper.dataset_colors('adult_restmovie')]
tasks_arm = ['aeronaut', 'mickey', 'rest']

plt.figure(figsize=(3,4))
sns.barplot(data=ide_df_arm, x='task', y='mean_IDE', palette=cmap_arm, edgecolor='k', linewidth=2, alpha=0.6)
sns.stripplot(data=ide_df_arm, x='task', y='mean_IDE', palette=cmap_arm, size=6, alpha=1, jitter=True, edgecolor='k', linewidth=1)
y_max = ide_df_arm['mean_IDE'].max()
for idx, (t1, t2) in enumerate([('aeronaut', 'rest')]):
    d1 = ide_df_arm[ide_df_arm['task'] == t1]['mean_IDE'].values
    d2 = ide_df_arm[ide_df_arm['task'] == t2]['mean_IDE'].values
    _, p_val = ttest_ind(d1, d2)
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'n.s.'
    y = y_max + 0.2 + idx * 0.4
    x1, x2 = tasks_arm.index(t1), tasks_arm.index(t2)
    plt.plot([x1, x2], [y, y], lw=1.5, c='k')
    plt.text((x1 + x2) * .5, y + 0.05, sig, ha='center', va='bottom', color='k', fontsize=10)
plt.ylabel('Mean IDE (across voxels)', fontsize=14)
plt.xlabel('Task', fontsize=14)
plt.title('Average IDE by Task', fontsize=16)
# sns.despine()
# plt.tight_layout()
# plt.savefig(f'{PLOT_DIR}/adult_restmovie_avg_ide_by_task_barplot.pdf', format='pdf', transparent=True)
# plt.show()

In [ ]:
# Adult Rest/Movie: composite plot — rest-aeronaut diff map + top ISC overlay
isc_img_arm = avg_results_arm.get('aeronaut', {}).get('ISC', None)
diff_img_arm = diff_results_arm.get('aeronaut_rest', {}).get('thresholded', None)

if isc_img_arm is not None and diff_img_arm is not None:
    isc_data_vol = isc_img_arm.get_fdata()
    isc_valid = isc_data_vol[~np.isnan(isc_data_vol)]
    threshold = 95
    isc_thr = np.nanpercentile(isc_valid, threshold)
    top_mask = (isc_data_vol >= isc_thr) & ~np.isnan(isc_data_vol)

    top_mask_vol = np.full(diff_img_arm.shape, np.nan, dtype=float)
    top_mask_vol[top_mask] = 1.0
    top_mask_img = nib.Nifti1Image(top_mask_vol, affine=diff_img_arm.affine, header=diff_img_arm.header)

    helper.generate_surface_plot(
        data_fn=[diff_img_arm, top_mask_img],
        image_fn=f'{PLOT_DIR}/adult_restmovie_aeronaut_rest_diff_top_isc_overlay.pdf',
        atlas='searchlight', cmap=helper.diverging_colormap_bp(), cbar_range=(-11, 11),
        surf_type='fslr', target_density='32k', include_cbar=True,
        title='Rest − Aeronaut IDE (thresholded) + Top ISC voxels',
        method='linear', threshold=None, alpha=0.5, mask_medial_wall=True,
        layers_kwargs=[
            dict(cmap=helper.diverging_colormap_bp(), cbar_range=(-11, 11), alpha=1,
                 label='ΔIDE (Rest − Aeronaut)', cbar=True),
            dict(cmap=ListedColormap([(1, 1, 0.6, 1.0)]), cbar_range=(0, 1), alpha=0.8,
                 label=f'Top {100 - threshold}% ISC mask', cbar=True),
        ],
    )
else:
    print("Warning: ISC or diff image not loaded — skipping composite plot")

# Section 3: PartlyCloudy

In [ ]:
results_df_pc = pd.read_csv(f'compiled/results/partlycloudy_compiled_results.csv')
results_df_pc.head()

In [ ]:
# PartlyCloudy: load results and participant info, define age groupings

def get_age_groupings_pc(age):
    if age < 5: return 'U05'
    elif age < 6: return 'U06'
    elif age < 8.2: return 'U082'
    elif age < 13: return 'U13'
    else: return 'Adult'

def get_motion_pc(subject_id):
    sub_data = par_df_pc[par_df_pc['participant_id'] == subject_id]
    if not sub_data.empty:
        return sub_data['movie_FD'].values[0]
    else:
        return np.nan


par_df = pd.read_csv('compiled/info/combined_participant_info.csv')
par_df_pc = par_df[par_df['dataset'] == 'PartlyCloudy'].reset_index(drop=True)
results_df_pc['AgeGroup'] = [get_age_groupings_pc(a) for a in results_df_pc['_age_raw']]
par_df_pc['Age']=par_df_pc['age_months']/12
par_df_pc['movie_FD'] = [get_motion_pc(sid) for sid in par_df_pc['participant_id']]

In [ ]:
# PartlyCloudy: demographics — age distribution and count by AgeGroup/Gender
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ax = axes[0]
par_df_pc['Age'].hist(bins=20, alpha=0.7, ax=ax, edgecolor='black')
par_df_pc['AgeGroup'] = par_df_pc['Age'].apply(get_age_groupings_pc)
ax.set_xlabel('Age (years)'); ax.set_ylabel('Count'); ax.set_title('Age Distribution')
ax.axvline(par_df_pc['Age'].mean(), color='red', linestyle='--', label=f'Mean: {par_df_pc["Age"].mean():.1f}')
ax.axvline(par_df_pc['Age'].median(), color='orange', linestyle='--', label=f'Median: {par_df_pc["Age"].median():.1f}')
ax.legend(); ax.grid(False)
ax = axes[1]
age_order_pc = ['U05', 'U06', 'U082', 'U13', 'Adult']
sns.countplot(data=par_df_pc, x='AgeGroup', hue='sex', order=age_order_pc,
              palette={'F': 'lightcoral', 'M': 'lightblue'}, ax=ax)
ax.set_xlabel('Age Group'); ax.set_ylabel('Count'); ax.set_title('Participant Count by Age Group and Gender')
ax.tick_params(axis='x', rotation=45)
sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
# PartlyCloudy: motion analysis — mean FD vs Age and by AgeGroup
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ax = axes[1]
mask = par_df_pc[['Age', 'movie_FD']].notnull().all(axis=1)
x, y = par_df_pc.loc[mask, 'Age'], par_df_pc.loc[mask, 'movie_FD']
sns.regplot(data=par_df_pc.loc[mask], x='Age', y='movie_FD', ax=ax, scatter_kws={'s': 20, 'alpha': 0.6}, ci=95)
r_fd, p_fd = stats.pearsonr(x, y)
ax.text(0.98, 0.98, f'n={mask.sum()}\nr={r_fd:.3f}\np={p_fd:.2e}', transform=ax.transAxes,
        ha='right', va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_title('Mean FD vs Age'); ax.set_ylabel('Mean Framewise Displacement (FD)')
ax = axes[0]
sns.barplot(data=par_df_pc, x='AgeGroup', y='movie_FD', order=age_order_pc,
            ax=ax, palette=helper.get_palette7_rainbow(), alpha=0.6, edgecolor='black')
sns.stripplot(data=par_df_pc, x='AgeGroup', y='movie_FD', order=age_order_pc,
              ax=ax, palette=helper.get_palette7_rainbow(), size=6, alpha=1)
age_group_counts_pc = par_df_pc['AgeGroup'].value_counts().to_dict()
for p, age_group in zip(ax.patches, age_order_pc):
    xpos = p.get_x() + p.get_width() / 2
    count = age_group_counts_pc.get(age_group, 0)
    ax.text(xpos, 0.01, f'n={count}', ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Age Group'); ax.set_ylabel('Mean FD'); ax.set_title('Motion (FD) by Age Group')
ax.tick_params(axis='x', rotation=45)
sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
# Create a 6-panel figure showing 
# Row 0: mean ISC by age group, mean IDE by age group, mean FD by age group
# Row 1: scatter plots of ISC , IDE, mean FD vs age, where regression lines
# are shown on the scatter plots, linear models used to predict ISC and IDE from age in months controlling for mean FD, and significance of age predictor is indicated on the scatter plots


# Create mean ISC dataframe
task_pc = 'pixar'
df_regional_isc = results_df_pc[(results_df_pc['task'] == task_pc) & (results_df_pc['measure'] == 'ISC')].copy()
single_isc = df_regional_isc.groupby('subject_id').mean(numeric_only=True).reset_index()
single_isc['AgeGroup'] = single_isc['_age_raw'].apply(get_age_groupings_pc)
age_order_pc = ['U05', 'U06', 'U082', 'U13', 'Adult']
single_isc['mean_FD'] = single_isc['subject_id'].map(par_df_pc.set_index('participant_id')['movie_FD'])

# Create mean IDE dataframe
df_regional_ide = results_df_pc[(results_df_pc['task'] == task_pc) & (results_df_pc['measure'] == 'TPHATE_DiffOp_IDE')].copy()
single_ide = df_regional_ide.groupby('subject_id').mean(numeric_only=True).reset_index()
single_ide['AgeGroup'] = single_ide['_age_raw'].apply(get_age_groupings_pc)
age_order_pc = ['U05', 'U06', 'U082', 'U13', 'Adult']
single_ide['mean_FD'] = single_ide['subject_id'].map(par_df_pc.set_index('participant_id')['movie_FD'])


fig, axes = plt.subplots(2, 3, figsize=(16, 10))
age_order_pc = ['U05', 'U06', 'U082', 'U13', 'Adult']
# Mean ISC by age group
ax = axes[0, 0]
sns.barplot(data=single_isc, x='AgeGroup', y='score', ax=ax, order=age_order_pc, palette='Set2', alpha=0.6, edgecolor='black')
sns.stripplot(data=single_isc, x='AgeGroup', y='score', ax=ax, order=age_order_pc, size=6, alpha=1, palette='Set2')
ax.set_ylabel('Mean ISC'); ax.set_title('Mean ISC by Age group')
# Mean IDE by age group
ax = axes[0, 1]
sns.barplot(data=single_ide, x='AgeGroup', y='score', ax=ax, order=age_order_pc, palette='Set2', alpha=0.6, edgecolor='black')
sns.stripplot(data=single_ide, x='AgeGroup', y='score', ax=ax, order=age_order_pc, size=6, alpha=1, palette='Set2')
ax.set_ylabel('Mean IDE'); ax.set_title('Mean IDE by Age group')

# Scatter plot of ISC vs age with regression line
ax = axes[1, 0]
sns.regplot(data=single_isc, x='age_months', y='score', ax=ax)

ols1 = smf.ols('score ~ age_months + mean_FD', data=single_isc).fit()

# extract beta and p-value for age_months predictor
beta_age_isc = ols1.params['age_months']
p_age_isc = ols1.pvalues['age_months']
ax.text(0.98, 0.02, f'n={len(single_isc)}\nbeta={beta_age_isc:.3e}\np={p_age_isc:.2f}', transform=ax.transAxes,
         ha='right', va='bottom', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlabel('Age (months)'); ax.set_ylabel('Mean ISC'); ax.set_title('Mean ISC ~ Age')

# Scatter plot of IDE vs age with regression line
ax = axes[1, 1]
sns.regplot(data=single_ide, x='age_months', y='score', ax=ax)
ols2 = smf.ols('score ~ age_months + mean_FD', data=single_ide).fit()
# extract beta and p-value for age_months predictor
beta_age_id = ols2.params['age_months']
p_age_id = ols2.pvalues['age_months']
ax.text(0.98, 0.02, f'n={len(single_ide)}\nbeta={beta_age_id:.3f}\np={p_age_id:.2f}', transform=ax.transAxes,
         ha='right', va='bottom', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlabel('Age (months)'); ax.set_ylabel('Mean IDE'); ax.set_title('Mean IDE ~ Age')
sns.despine(); plt.tight_layout();

# Mean FD by age group
ax = axes[0, 2]
sns.barplot(data=single_ide, x='AgeGroup', y='mean_FD', ax=ax, order=age_order_pc, palette='Set2', alpha=0.6, edgecolor='black')
sns.stripplot(data=single_ide, x='AgeGroup', y='mean_FD', ax=ax, order=age_order_pc, size=6, alpha=1, palette='Set2')
ax.set_ylabel('Mean FD'); ax.set_title('Mean FD by Age group')

# Scatter plot of ISC vs age with regression line
ax = axes[1, 2]
sns.regplot(data=single_isc, x='age_months', y='mean_FD', ax=ax)

ols3 = smf.ols('mean_FD ~ age_months', data=single_isc).fit()

# extract beta and p-value for age_months predictor
beta_age_fd = ols3.params['age_months']
p_age_fd = ols3.pvalues['age_months']
ax.text(0.98, 0.02, f'n={len(single_isc)}\nbeta={beta_age_fd:.3e}\np={p_age_fd:.2f}', transform=ax.transAxes,
         ha='right', va='bottom', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlabel('Age (months)'); ax.set_ylabel('Mean FD'); ax.set_title('Mean FD ~ Age')
plt.savefig(f'{PLOT_DIR}/partlycloudy_means_by_age.pdf', format='pdf', transparent=True); plt.show()


In [ ]:
# PartlyCloudy: surface grid by AgeGroup (ISC and IDE)
results_dfa_pc = results_df_pc[results_df_pc['task'] == task_pc]
df_pc_surf = results_dfa_pc[results_dfa_pc['measure'].isin(['ISC', 'TPHATE_DiffOp_IDE'])].copy()
mean_by_pc = df_pc_surf.groupby(['AgeGroup', 'measure', 'region_name'])['score'].agg(
    lambda x: x.mean(skipna=True)).reset_index()

data_arrs_pc, titles_pc, img_fns_pc = [], [], []
for m in ['ISC', 'TPHATE_DiffOp_IDE']:
    for age in age_order_pc:
        s = get_average_results(mean_by_pc, col_names=['AgeGroup', 'measure'], target_names=[age, m], value_name='score', region_name_order=atlas_labels)
        data_arrs_pc.append(s)
        titles_pc.append(f"{age} {'ISC' if m == 'ISC' else 'IDE'}")
        img_fns_pc.append(f'PartlyCloudy/plots/{task_pc}_{age}_{m}.png')

helper.compile_surface_plots_to_grid_by_rows(
    img_fns_pc, data_arrs_pc,
    output_path=f'{PLOT_DIR}/partlycloudy_agegroup_isc_ide_surface_grid.pdf',
    n_rows=2, surf_type='fslr', target_density='32k', method='linear',
    atlas='Schaefer', rerun=True,
    cmaps_per_row=[ISC_CMAP, ID_CMAP],
    cbar_ranges_per_row=[[0.0, 0.3], [1, 16]],
    cbar_labels_per_row=['ISC', 'IDE'],
    titles=titles_pc
)

In [ ]:
# PartlyCloudy: ISC-IDE permutation correlation per subject
df_corr_pc = pd.read_csv(f'compiled/results/combined_corrs.csv')
df_corr_pc = df_corr_pc[df_corr_pc['dataset']=='PartlyCloudy'].reset_index(drop=True)
print(f"ISC-IDE correlation computed for {len(df_corr_pc)} subjects")
df_corr_pc['AgeGroup'] = [get_age_groupings_pc(a/12) for a in df_corr_pc['age_months']]
df_corr_pc.head()

In [ ]:
# PartlyCloudy: ISC-IDE z-score vs Age scatter
plt.figure(figsize=(8, 6))
sns.regplot(data=df_corr_pc, x='age_months', y='zscore', scatter_kws={'s': 50, 'alpha': 0.7},
            line_kws={'color': 'k', 'linewidth': 2})
r_overall, p_overall = stats.pearsonr(df_corr_pc['age_months'], df_corr_pc['zscore'])
plt.title('Whole-brain ISC-IDE correlation vs Age')
plt.text(0.98, 0.98, f'n={len(df_corr_pc)}\nr={r_overall:.3f}\np={p_overall:.2e}',
         transform=plt.gca().transAxes, ha='right', va='top',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
plt.xlabel('Age (years)'); plt.ylabel('ISC-IDE correlation (z-score)')
sns.despine(); plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/partlycloudy_isc_ide_corr_vs_age_scatter.pdf', format='pdf', transparent=True)
plt.show()

In [ ]:
# combined FD with results_df_pc at subject level
temp = pd.read_csv(f'compiled/results/combined_corr_analyses.csv' )
temp_pc = temp[temp['dataset']=='partlycloudy']
results_df_pc['mean_FD'] = results_df_pc['subject_id'].map(temp_pc.set_index('subject')['movie_FD'])
results_df_pc['sex'] = results_df_pc['subject_id'].map(temp_pc.set_index('subject')['sex'])
REGION_ORDER_PC = results_df_pc['region_name'].values[:400]
dataframe_pc = results_df_pc.pivot_table(
    index=['subject_id', 'region_name', 'task','age_months','_age_raw', 'mean_FD', 'sex'],
    columns='measure', values='score').reset_index()
dataframe_pc.head()


# Section 4: HBN (Healthy Brain Network)

In [ ]:
# HBN: load results and participant info
results_df_hbn =  pd.read_csv(f'compiled/results/hbn_compiled_results.csv')

par_df_hbn = pd.read_csv('HBN/basic_cohort_info.csv', index_col=1)
par_df_hbn['include_both_tasks'] = [
    1 if (int(np.isnan(row.movie_FD)) + int(np.isnan(row.rest_FD)) + int(row.cleaning_done == 0) == 0) else 0
    for _, row in par_df_hbn.iterrows()]
par_df_hbn = par_df_hbn[par_df_hbn['include_both_tasks'] == 1]
REGION_ORDER_HBN = results_df_hbn['region_name'].values[:400]

# par_df_hbn.index = par_df_hbn.subject_id
print(f"HBN subjects: {par_df_hbn.shape[0]}, results rows: {results_df_hbn.shape[0]}")

def age_group_hbn(age):
    # Figure out which age bin falls into
    for i in range(len(hc.AGE_BINS)):
        if age < hc.AGE_BINS[i]:
            return hc.HBN_AGE_GROUPS[i]
    return None

def get_mean_fd(subject, task):
    if task == 'movieTP':
        fd_col = 'movie_FD'
    else:
        fd_col = 'rest_FD'
    if subject in par_df_hbn.index and fd_col in par_df_hbn.columns:
        return par_df_hbn.loc[subject, fd_col], par_df_hbn.loc[subject, 'sex']
    else:
        return np.nan
    
def get_par_info(subject):
    row = par_df[par_df['participant_id']==subject]
    return row.movie_FD.item(), row.rest_FD.item(), row.sex.item()




In [ ]:
from statsmodels.stats.anova import anova_lm
results_df_hbn['log_age']= np.log(results_df_hbn['age_months']+1)

# HBN: ANOVA AgeGroup x task for IDE
results_df_hbn = results_df_hbn[results_df_hbn['measure'] == 'TPHATE_DiffOp_IDE']
temp_hbn = results_df_hbn.groupby(['subject_id', 'task', 'age_months','_age_raw']).mean(numeric_only=True).reset_index()
# Merge in the FD and sex on subject_id and task
vals = np.array([get_mean_fd(row['subject_id'], row['task']) for _, row in temp_hbn.iterrows()])
temp_hbn['sex']=vals[:, 1]
temp_hbn['mean_FD']=vals[:, 0]
temp_hbn['mean_FD'] = temp_hbn['mean_FD'].astype(float)
temp_hbn['task'] = temp_hbn['task'].astype('category')
temp_hbn['sex'] = temp_hbn['sex'].astype('category')
Q=6
# Bin into Q bins
temp_hbn['AgeGroup_Binned'] = pd.qcut(
    temp_hbn['_age_raw'],
    q=Q,
    retbins=False
).apply(lambda x: f'{np.min(x)},{np.max(x)}')  # midpoint of each bin interval
# Now relabel the bins to be more concise
bin_edges = pd.qcut(temp_hbn['_age_raw'], q=Q, retbins=True)[1]
bin_labels = []
for i in range(len(bin_edges) - 1):
    lower = int(bin_edges[i])
    upper = int(bin_edges[i + 1])
    bin_labels.append(f'{lower}-{upper}')
temp_hbn['AgeGroup_Binned'] = pd.qcut(
    temp_hbn['_age_raw'],
    q=Q,
    retbins=False,
    labels=bin_labels
)
print("Age bins and their ranges:")
for label in bin_labels:
    bin_range = temp_hbn[temp_hbn['AgeGroup_Binned'] == label]['_age_raw'].agg(['min', 'max'])
    count = len(temp_hbn[temp_hbn['AgeGroup_Binned'] == label])/2
    print(f"{label}: n={count}, {bin_range['min']:.1f} - {bin_range['max']:.1f} years")

temp_hbn['AgeGroup_Binned'] = temp_hbn['AgeGroup_Binned'].astype('category')
temp_hbn['task'] = temp_hbn['task'].astype('category')

model = smf.ols('score ~ age_months * C(task) + sex + mean_FD', data=temp_hbn).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print(model.summary())
print(anova_table)

# # Run post-hoc tests to compare the groups
# from statsmodels.stats.multicomp import pairwise_tukeyhsd
# posthoc = pairwise_tukeyhsd(temp_hbn['score'], temp_hbn['AgeGroup_Binned'] + '_' + temp_hbn['task'])
# print(posthoc)


In [ ]:

# Create mean ISC dataframe for HBN
df_regional_isc_hbn = results_df_hbn[(results_df_hbn['task'] == 'movieTP') & (results_df_hbn['measure'] == 'ISC')].copy()
df_regional_isc_hbn['age_y'] = df_regional_isc_hbn['age_months']/12
single_isc_hbn = df_regional_isc_hbn.groupby('subject_id').mean(numeric_only=True).reset_index()
single_isc_hbn['AgeGroup_Binned'] = pd.qcut(single_isc_hbn['_age_raw'], q=Q, retbins=False, labels=bin_labels)
single_isc_hbn['mean_FD'] = single_isc_hbn['subject_id'].map(par_df_hbn['movie_FD'])
single_isc_hbn['sex'] = single_isc_hbn['subject_id'].map(par_df_hbn['sex'])

# Create mean IDE dataframe for HBN
df_regional_ide_hbn = results_df_hbn[(results_df_hbn['task'] == 'movieTP') & (results_df_hbn['measure'] == 'TPHATE_DiffOp_IDE')].copy()
df_regional_ide_hbn['age_y'] = df_regional_ide_hbn['age_months']/12
single_ide_hbn = df_regional_ide_hbn.groupby('subject_id').mean(numeric_only=True).reset_index()
single_ide_hbn['AgeGroup_Binned'] = pd.qcut(single_ide_hbn['_age_raw'], q=Q, retbins=False, labels=bin_labels)
single_ide_hbn['mean_FD'] = single_ide_hbn['subject_id'].map(par_df_hbn['movie_FD'])
single_ide_hbn['sex'] = single_ide_hbn['subject_id'].map(par_df_hbn['sex'])

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Mean ISC by age group
ax = axes[0, 0]
sns.barplot(data=single_isc_hbn, x='AgeGroup_Binned', y='score', ax=ax, order=bin_labels, palette='Set3', alpha=0.6, edgecolor='black')
sns.stripplot(data=single_isc_hbn, x='AgeGroup_Binned', y='score', ax=ax, order=bin_labels, size=4, alpha=0.7, palette='Set3')
ax.set_ylabel('Mean ISC'); ax.set_title('Mean ISC by Age group (HBN)')
ax.tick_params(axis='x', rotation=45)

# Mean IDE by age group
ax = axes[0, 1]
sns.barplot(data=single_ide_hbn, x='AgeGroup_Binned', y='score', ax=ax, order=bin_labels, palette='Set3', alpha=0.6, edgecolor='black')
sns.stripplot(data=single_ide_hbn, x='AgeGroup_Binned', y='score', ax=ax, order=bin_labels, size=4, alpha=0.7, palette='Set3')
ax.set_ylabel('Mean IDE'); ax.set_title('Mean IDE by Age group (HBN)')
ax.tick_params(axis='x', rotation=45)

# Scatter plot of ISC vs age with regression line
ax = axes[1, 0]
sns.regplot(data=single_isc_hbn, x='age_y', y='score', ax=ax)
ols1_hbn = smf.ols('score ~ age_y + mean_FD', data=single_isc_hbn).fit()
beta_age_isc_hbn = ols1_hbn.params['age_y']
p_age_isc_hbn = ols1_hbn.pvalues['age_y']
ax.text(0.98, 0.98, f'n={len(single_isc_hbn)}\nbeta={beta_age_isc_hbn:.3e}\np={p_age_isc_hbn:.2e}', 
    transform=ax.transAxes, ha='right', va='top', 
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlabel('Age (years)'); ax.set_ylabel('Mean ISC'); ax.set_title('Mean ISC ~ Age (HBN)')

# Scatter plot of IDE vs age with regression line
ax = axes[1, 1]
sns.regplot(data=single_ide_hbn, x='age_y', y='score', ax=ax)
ols2_hbn = smf.ols('score ~ age_y + mean_FD', data=single_ide_hbn).fit()
beta_age_id_hbn = ols2_hbn.params['age_y']
p_age_id_hbn = ols2_hbn.pvalues['age_y']
ax.text(0.98, 0.98, f'n={len(single_ide_hbn)}\nbeta={beta_age_id_hbn:.3e}\np={p_age_id_hbn:.2e}', 
    transform=ax.transAxes, ha='right', va='top', 
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlabel('Age (years)'); ax.set_ylabel('Mean IDE'); ax.set_title('Mean IDE ~ Age (HBN)')

# Mean FD by age group
ax = axes[0, 2]
sns.barplot(data=single_ide_hbn, x='AgeGroup_Binned', y='mean_FD', ax=ax, order=bin_labels, palette='Set3', alpha=0.6, edgecolor='black')
sns.stripplot(data=single_ide_hbn, x='AgeGroup_Binned', y='mean_FD', ax=ax, order=bin_labels, size=4, alpha=0.7, palette='Set3')
ax.set_ylabel('Mean FD'); ax.set_title('Mean FD by Age group (HBN)')
ax.tick_params(axis='x', rotation=45)

# Scatter plot of mean FD vs age with regression line
ax = axes[1, 2]
sns.regplot(data=single_isc_hbn, x='age_y', y='mean_FD', ax=ax)
ols3_hbn = smf.ols('mean_FD ~ age_y', data=single_isc_hbn).fit()
beta_age_fd_hbn = ols3_hbn.params['age_y']
p_age_fd_hbn = ols3_hbn.pvalues['age_y']
ax.text(0.98, 0.98, f'n={len(single_isc_hbn)}\nbeta={beta_age_fd_hbn:.3e}\np={p_age_fd_hbn:.2e}', 
    transform=ax.transAxes, ha='right', va='top', 
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlabel('Age (years)'); ax.set_ylabel('Mean FD'); ax.set_title('Mean FD ~ Age (HBN)')

sns.despine(); plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/hbn_means_by_age.pdf', format='pdf', transparent=True)
plt.show()
par_df_hbn = par_df_hbn.reset_index()

In [ ]:
import matplotlib.pyplot as plt

# Set Arial as the default font
plt.rcParams['font.family'] = 'Arial'
# par_df_hbn = par_df_hbn.reset_index()
# Pool together the partly cloudy and healthy brain network datasets to run a mega-analysis of age effects on ISC and IDE across the full developmental span,
# controlling for dataset, mean FD, and sex
# First, we need to add a 'dataset' column to each dataframe and then concatenate them
single_isc_pc = single_isc.copy()
single_isc_pc['dataset'] = 'PartlyCloudy'
single_ide_pc = single_ide.copy()
single_ide_pc['dataset'] = 'PartlyCloudy'
single_isc_hbn_copy = single_isc_hbn.copy()
single_isc_hbn_copy['dataset'] = 'HBN'
single_ide_hbn_copy = single_ide_hbn.copy()
single_ide_hbn_copy['dataset'] = 'HBN'

single_isc_pc['sex'] = single_isc_pc['subject_id'].map(par_df_pc.set_index('participant_id')['sex'])
single_ide_pc['sex'] = single_ide_pc['subject_id'].map(par_df_pc.set_index('participant_id')['sex'])
single_ide_hbn_copy['sex'] = single_ide_hbn_copy['subject_id'].map(par_df_hbn.set_index('subject_id')['sex'])
single_isc_hbn_copy['sex'] = single_isc_hbn_copy['subject_id'].map(par_df_hbn.set_index('subject_id')['sex'])


single_isc_combined = pd.concat([single_isc_pc, single_isc_hbn_copy], ignore_index=True)
single_ide_combined = pd.concat([single_ide_pc, single_ide_hbn_copy], ignore_index=True)
single_isc_combined['age_y'] = single_isc_combined['age_months'] / 12
single_ide_combined['age_y'] = single_ide_combined['age_months'] / 12

# Now we can run regression analyses to predict ISC and IDE from age, controlling for dataset, mean FD, and sex
# Use a LME model with random intercepts for dataset
lme_isc_combined = smf.mixedlm('score ~ age_y + mean_FD + sex', data=single_isc_combined, groups=single_isc_combined['dataset']).fit()
lme_ide_combined = smf.mixedlm('score ~ age_y + mean_FD + sex', data=single_ide_combined, groups=single_ide_combined['dataset']).fit()
print("Combined ISC regression results:")
print(lme_isc_combined.summary())
print("\nCombined IDE regression results:")
print(lme_ide_combined.summary())

# Plot the combined data with regression lines and betas and p-values for the age predictor
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Create color palette for datasets
dataset_palette = {
    'PartlyCloudy': helper.dataset_colors('partlycloudy'),
    'HBN': helper.dataset_colors('hbn')
}

# ISC plot
ax = axes[0]
sns.scatterplot(data=single_isc_combined, x='age_y', y='score', hue='dataset', 
                palette=dataset_palette, s=24, ax=ax, legend=True, edgecolor='k', linewidth=0.1)

sns.regplot(data=single_isc_combined, x='age_y', y='score', scatter=False, ax=ax, color='gray',line_kws={'linewidth': 2})
beta_age_isc_combined = lme_isc_combined.params['age_y']
p_age_isc_combined = lme_isc_combined.pvalues['age_y']
sig_isc = helper.get_asterisks_pvalue(p_age_isc_combined)
# Compute rho value for scatter of ISC vs agemonths
rho_isc, p_rho_isc = stats.spearmanr(single_isc_combined['age_y'], single_isc_combined['score'])
ax.text(0.92, 0.64, f'β={beta_age_isc_combined:.3f}{sig_isc}\nρ={rho_isc:.2f}', 
    transform=ax.transAxes, ha='right', va='top', fontsize=12)
ax.set_xlabel('Age (years)', fontsize=12); ax.set_ylabel('ISC', fontsize=12); ax.set_title('Average ISC ~ age + FD + (1|dataset)')

# IDE plot
ax = axes[1]
sns.scatterplot(data=single_ide_combined, x='age_y', y='score', hue='dataset',  
                palette=dataset_palette, s=24,  ax=ax, legend=True, edgecolor='k', linewidth=0.1)
sns.regplot(data=single_ide_combined, x='age_y', y='score', scatter=False, ax=ax, color='gray', line_kws={'linewidth': 2})
beta_age_ide_combined = lme_ide_combined.params['age_y']
p_age_ide_combined = lme_ide_combined.pvalues['age_y']
rho_ide, p_rho_ide = stats.spearmanr(single_ide_combined['age_y'], single_ide_combined['score'])
sig_ide = helper.get_asterisks_pvalue(p_age_ide_combined)
ax.text(0.98, 0.4, f'β={beta_age_ide_combined:.3f}{sig_ide}\nρ={rho_ide:.2f}', 
    transform=ax.transAxes, ha='right', va='top',  fontsize=12)
ax.set_xlabel('Age (years)', fontsize=12); ax.set_ylabel('Dimensionality', fontsize=12); ax.set_title('Average ID ~ age + FD + (1|dataset)')

sns.despine(); plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/combined_age_effects_isc_ide.pdf', format='pdf', transparent=True)
plt.show()

In [ ]:
results_df_nar.head()

In [ ]:
import matplotlib.pyplot as plt

# Set Arial as the default font
plt.rcParams['font.family'] = 'Arial'
# par_df_hbn = par_df_hbn.reset_index()
# Pool together the partly cloudy and healthy brain network datasets to run a mega-analysis of age effects on ISC and IDE across the full developmental span,
# controlling for dataset, mean FD, and sex
# First, we need to add a 'dataset' column to each dataframe and then concatenate them
single_isc_pc = single_isc.copy()
single_isc_pc['dataset'] = 'PartlyCloudy'
single_ide_pc = single_ide.copy()
single_ide_pc['dataset'] = 'PartlyCloudy'
single_isc_hbn_copy = single_isc_hbn.copy()
single_isc_hbn_copy['dataset'] = 'HBN'
single_ide_hbn_copy = single_ide_hbn.copy()
single_ide_hbn_copy['dataset'] = 'HBN'

single_isc_pc['sex'] = single_isc_pc['subject_id'].map(par_df_pc.set_index('participant_id')['sex'])
single_ide_pc['sex'] = single_ide_pc['subject_id'].map(par_df_pc.set_index('participant_id')['sex'])
single_ide_hbn_copy['sex'] = single_ide_hbn_copy['subject_id'].map(par_df_hbn.set_index('subject_id')['sex'])
single_isc_hbn_copy['sex'] = single_isc_hbn_copy['subject_id'].map(par_df_hbn.set_index('subject_id')['sex'])


single_isc_combined = pd.concat([single_isc_pc, single_isc_hbn_copy], ignore_index=True)
single_ide_combined = pd.concat([single_ide_pc, single_ide_hbn_copy], ignore_index=True)
single_isc_combined['age_y'] = single_isc_combined['age_months'] / 12
single_ide_combined['age_y'] = single_ide_combined['age_months'] / 12

# Now we can run regression analyses to predict ISC and IDE from age, controlling for dataset, mean FD, and sex
# Use a LME model with random intercepts for dataset
lme_isc_combined = smf.mixedlm('score ~ age_y + mean_FD + sex', data=single_isc_combined, groups=single_isc_combined['dataset']).fit()
lme_ide_combined = smf.mixedlm('score ~ age_y + mean_FD + sex', data=single_ide_combined, groups=single_ide_combined['dataset']).fit()
print("Combined ISC regression results:")
print(lme_isc_combined.summary())
print("\nCombined IDE regression results:")
print(lme_ide_combined.summary())

# Plot the combined data with regression lines and betas and p-values for the age predictor
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Create color palette for datasets
dataset_palette = {
    'PartlyCloudy': helper.dataset_colors('partlycloudy'),
    'HBN': helper.dataset_colors('hbn')
}

# ISC plot
ax = axes[0]
sns.scatterplot(data=single_isc_combined, x='age_y', y='score', hue='dataset', 
                palette=dataset_palette, s=24, ax=ax, legend=True, edgecolor='k', linewidth=0.1)

sns.regplot(data=single_isc_combined, x='age_y', y='score', scatter=False, ax=ax, color='gray',line_kws={'linewidth': 2})
beta_age_isc_combined = lme_isc_combined.params['age_y']
p_age_isc_combined = lme_isc_combined.pvalues['age_y']
sig_isc = helper.get_asterisks_pvalue(p_age_isc_combined)
# Compute rho value for scatter of ISC vs agemonths
rho_isc, p_rho_isc = stats.spearmanr(single_isc_combined['age_y'], single_isc_combined['score'])
ax.text(0.92, 0.64, f'β={beta_age_isc_combined:.3f}{sig_isc}\nρ={rho_isc:.2f}', 
    transform=ax.transAxes, ha='right', va='top', fontsize=12)
ax.set_xlabel('Age (years)', fontsize=12); ax.set_ylabel('ISC', fontsize=12); ax.set_title('Average ISC ~ age + FD + (1|dataset)')

# IDE plot
ax = axes[1]
sns.scatterplot(data=single_ide_combined, x='age_y', y='score', hue='dataset',  
                palette=dataset_palette, s=24,  ax=ax, legend=True, edgecolor='k', linewidth=0.1)
sns.regplot(data=single_ide_combined, x='age_y', y='score', scatter=False, ax=ax, color='gray', line_kws={'linewidth': 2})
beta_age_ide_combined = lme_ide_combined.params['age_y']
p_age_ide_combined = lme_ide_combined.pvalues['age_y']
rho_ide, p_rho_ide = stats.spearmanr(single_ide_combined['age_y'], single_ide_combined['score'])
sig_ide = helper.get_asterisks_pvalue(p_age_ide_combined)
ax.text(0.98, 0.4, f'β={beta_age_ide_combined:.3f}{sig_ide}\nρ={rho_ide:.2f}', 
    transform=ax.transAxes, ha='right', va='top',  fontsize=12)
ax.set_xlabel('Age (years)', fontsize=12); ax.set_ylabel('Dimensionality', fontsize=12); ax.set_title('Average ID ~ age + FD + (1|dataset)')

sns.despine(); plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/combined_age_effects_isc_ide.pdf', format='pdf', transparent=True)
plt.show()

In [ ]:
# Print out the betas, z values, confidence intervals, and p-values for the age predictor in both models
beta_isc, p_isc, ci_isc, z_isc = lme_isc_combined.params['age_y'], lme_isc_combined.pvalues['age_y'], lme_isc_combined.conf_int().loc['age_y'].values, lme_isc_combined.tvalues['age_y']
beta_ide, p_ide, ci_ide, z_ide = lme_ide_combined.params['age_y'], lme_ide_combined.pvalues['age_y'], lme_ide_combined.conf_int().loc['age_y'].values, lme_ide_combined.tvalues['age_y']

print(f'ISC regression: β={beta_isc:.2e}, 95% CI=({ci_isc[0]:.2e}, {ci_isc[1]:.2e}), z={z_isc:.2f}, p={p_isc:.2e}, {helper.get_asterisks_pvalue(p_isc)}')
print(f'IDE regression: β={beta_ide:.2e}, 95% CI=({ci_ide[0]:.2e}, {ci_ide[1]:.2e}), z={z_ide:.2f}, p={p_ide:.2e}, {helper.get_asterisks_pvalue(p_ide)}')


# Infant adult comparison: ISC and IDE 

In [ ]:
arm = par_df[par_df['dataset']=='AdultRestMovie']
a_movie_fd = arm['movie_FD'].values[:-1]
a_rest_fd = arm['rest_FD'].values[:-1]
a_sex = arm['sex'].values[:-1]
irm = par_df[par_df['dataset']=='InfantRestMovie']
i_movie_fd = [i for i in irm['movie_FD'].values if i != 0]
i_rest_fd = [i for i in irm['rest_FD'].values if i != 0]
i_sex = irm['sex'].values
i_sex.shape, len(i_rest_fd), len(i_movie_fd), len(a_sex), len(a_rest_fd), len(a_movie_fd) 


In [ ]:
# from matplotlib.patches import Patch

# # Load in the infant and adult data for rest and movie
measure = 'TPHATE_DiffOp_IDE'
adult_aero = f'{results_dir_arm}/aeronaut_{measure}_all_subjects_results.nii.gz'
adult_rest = f'{results_dir_arm}/rest_{measure}_all_subjects_results.nii.gz'
adult_mask = NiftiMasker(mask_img=f'adult_restmovie/masks/adult_restmovie_intersect_mask.nii.gz').fit()
adult_aero_data = adult_mask.transform(adult_aero).mean(axis=1)
adult_rest_data = adult_mask.transform(adult_rest).mean(axis=1)
print(f"Loaded adult data: {adult_aero_data.shape[0]} subjects, {adult_rest_data.shape[0]} subjects")

inf_aero = f'infant_restmovie/results/aeronaut_{measure}_all_subjects_results.nii.gz'
inf_aero_mask = NiftiMasker(mask_img=f'infant_restmovie/masks/InfantRestMovie_intersect_mask_aeronaut.nii.gz').fit()
inf_aero_data = inf_aero_mask.transform(inf_aero).mean(axis=1)
inf_sleep = f'infant_restmovie/results/sleep_{measure}_all_subjects_results.nii.gz'
inf_sleep_mask = NiftiMasker(mask_img=f'infant_restmovie/masks/InfantRestMovie_intersect_mask_sleep.nii.gz').fit()
inf_sleep_data = inf_sleep_mask.transform(inf_sleep).mean(axis=1)
print(f"Loaded infant data: {inf_aero_data.shape[0]} subjects, {inf_sleep_data.shape[0]} subjects")


# add these means to a combined dataframe for plotting
subject_means = pd.DataFrame({
    'dataset': ['infant_restmovie'] * (len(inf_aero_data) + len(inf_sleep_data)) + ['adult_restmovie'] * (len(adult_aero_data) + len(adult_rest_data)),
    'task': ['aeronaut'] * len(inf_aero_data) + ['rest'] * len(inf_sleep_data) + ['aeronaut'] * len(adult_aero_data) + ['rest'] * len(adult_rest_data),
    'subject_id': list(range(len(inf_aero_data))) + list(range(len(inf_aero_data), len(inf_sleep_data)+len(inf_aero_data))) + list(range(len(adult_aero_data))) + list(range(len(adult_rest_data))),
    'FD': np.concatenate([i_movie_fd, i_rest_fd , a_movie_fd , a_rest_fd]),
     'sex':np.concatenate([i_sex, a_sex, a_sex]), 
    'score': np.concatenate([inf_aero_data, inf_sleep_data, adult_aero_data, adult_rest_data])
})


In [ ]:
adult_rest

In [ ]:
ttest_rel( adult_rest, adult_aero, alternative='two-sided')

In [ ]:
ttest_ind(inf_aero, inf_sleep, alternative='two-sided', equal_var=False)

In [ ]:
# Use a 2=way anova to test for main effects of dataset and task, and their interaction
import statsmodels.api as sm
from statsmodels.formula.api import ols

model = ols('score ~ C(dataset) * C(task) ', data=subject_means).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print(model.summary())
print(anova_table)

# Run post-hoc tests to compare the groups
from statsmodels.stats.multicomp import pairwise_tukeyhsd
posthoc = pairwise_tukeyhsd(subject_means['score'], subject_means['dataset'] + '_' + subject_means['task'])
print(posthoc)


In [ ]:
from matplotlib.patches import Patch
from scipy.stats import ttest_ind, ttest_rel, spearmanr

infant_color = helper.dataset_colors('infant_restmovie')
adult_color  = helper.dataset_colors('adult_restmovie')
hbn_color    = helper.dataset_colors('hbn')

rest_keywords = {'rest', 'sleep'}
def is_rest_task(task):
    return any(k in task.lower() for k in rest_keywords)

def sig_label(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'n.s.'

def add_bracket(ax, x1, x2, y, label, tick_h=0.15):
    ax.plot([x1, x1, x2, x2], [y, y + tick_h, y + tick_h, y], lw=1.0, c='k', clip_on=False)
    ax.text((x1 + x2) / 2, y + tick_h + 0.02, label,
            ha='center', va='bottom', fontsize=7.5, clip_on=False)

# --- Infant / Adult setup ---
datasets    = ['infant_restmovie', 'adult_restmovie']
ds_colors   = {'infant_restmovie': infant_color, 'adult_restmovie': adult_color}
tasks_by_ds = {ds: sorted(subject_means[subject_means['dataset'] == ds]['task'].unique(),
                           key=lambda t: (1 if is_rest_task(t) else 0, t))
               for ds in datasets}

# --- HBN setup ---
hbn_means  = temp_hbn #results_df_hbn.groupby(['subject_id','task','AgeGroup'])['score'].mean().reset_index()
hbn_tasks  = sorted(hbn_means['task'].unique(), key=lambda t: (1 if is_rest_task(t) else 0, t))
order_hbn = ['5-8', '8-10','10-11', '11-13', '13-16', '16-21', ]
bar_width = 0.3
gap       = 0.4
hbn_x_start   = len(datasets) + gap
hbn_x_centers = [hbn_x_start + j for j in range(len(order_hbn))]

fig, ax = plt.subplots(figsize=(8, 5))

# ── Infant / Adult bars ────────────────────────────────────────────────────────
group_bars = {}   # ds -> {task -> {pos, mean, sem, scores}}
for i, ds in enumerate(datasets):
    tasks   = tasks_by_ds[ds]
    n       = len(tasks)
    offsets = np.linspace(-(n - 1) * bar_width / 2, (n - 1) * bar_width / 2, n)
    color   = ds_colors[ds]
    group_bars[ds] = {}
    for task, offset in zip(tasks, offsets):
        pos    = i + offset
        df_t   = subject_means[(subject_means['dataset'] == ds) & (subject_means['task'] == task)]
        mean_v = df_t['score'].mean()
        sem_v  = df_t['score'].sem()
        hatch  = '///' if is_rest_task(task) else None
        ax.bar(pos, mean_v, width=bar_width, color=color, hatch=hatch,
               edgecolor='k', linewidth=1.5, alpha=0.7,
               yerr=sem_v, capsize=0, error_kw={'elinewidth': 1.5, 'capthick': 1.5})
        jitter = np.random.normal(0, bar_width * 0.08, len(df_t))
        ax.scatter(pos + jitter, df_t['score'], color=color,
                   s=15, alpha=0.8, edgecolors='k', linewidth=0.3, zorder=5)
        group_bars[ds][task] = dict(pos=pos, mean=mean_v, sem=sem_v, scores=df_t['score'].values)

# ── Infant / Adult statistics ──────────────────────────────────────────────────
for ds in datasets:
    info  = group_bars[ds]
    movie = next((t for t in info if not is_rest_task(t)), None)
    rest  = next((t for t in info if     is_rest_task(t)), None)
    if not movie or not rest:
        continue
    s_m, s_r = info[movie]['scores'], info[rest]['scores']
    if ds == 'infant_restmovie':
        _, pval = ttest_ind(s_m, s_r)          # independent: different subjects
    else:
        n = min(len(s_m), len(s_r))
        _, pval = ttest_rel(s_m[:n], s_r[:n])  # paired: same subjects, same order
    y_top = 26
    add_bracket(ax, info[movie]['pos'], info[rest]['pos'], y_top + 0.2, sig_label(pval))

# ── HBN bars ───────────────────────────────────────────────────────────────────
hbn_bin_info = []   # for age-trend regression
for j, ag in enumerate(order_hbn):
    df_ag   = hbn_means[hbn_means['AgeGroup_Binned'] == ag]
    n       = len(hbn_tasks)
    offsets = np.linspace(-(n - 1) * bar_width / 2, (n - 1) * bar_width / 2, n)
    bin_data = {}
    for task, offset in zip(hbn_tasks, offsets):
        df_t = df_ag[df_ag['task'] == task]
        if df_t.empty:
            continue
        pos    = hbn_x_centers[j] + offset
        mean_v = df_t['score'].mean()
        sem_v  = df_t['score'].sem()
        hatch  = '///' if is_rest_task(task) else None
        ax.bar(pos, mean_v, width=bar_width, color=hbn_color, hatch=hatch,
               edgecolor='k', linewidth=1.5, alpha=0.7,
               yerr=sem_v, capsize=3, error_kw={'elinewidth': 1.5, 'capthick': 1.5})
        jitter = np.random.normal(0, bar_width * 0.08, len(df_t))
        ax.scatter(pos + jitter, df_t['score'], color=hbn_color,
                   s=10, alpha=0.8, edgecolors='k', linewidth=0.3, zorder=5)
        bin_data[task] = dict(pos=pos, mean=mean_v, sem=sem_v,
                              scores=df_t.set_index('subject_id')['score'])
    
    # Paired t-test (same subjects did both tasks within each age bin)
    movie_t = next((t for t in bin_data if not is_rest_task(t)), None)
    rest_t  = next((t for t in bin_data if     is_rest_task(t)), None)
    if movie_t and rest_t:
        common = bin_data[movie_t]['scores'].index.intersection(bin_data[rest_t]['scores'].index)
        if len(common) >= 5:
            _, pval = ttest_rel(bin_data[movie_t]['scores'][common],
                                bin_data[rest_t]['scores'][common])
            y_top = 26
            add_bracket(ax, bin_data[movie_t]['pos'], bin_data[rest_t]['pos'],
                        y_top + 0.2, sig_label(pval))
        hbn_bin_info.append(dict(x=hbn_x_centers[j],
                                 movie=bin_data[movie_t]['mean'],
                                 rest=bin_data[rest_t]['mean']))


# ── Axes ───────────────────────────────────────────────────────────────────────
ax.set_xticks(list(range(len(datasets))) + hbn_x_centers)
ax.set_xticklabels(['Infant', 'Adult'] + order_hbn, fontsize=9.5)
ax.set_ylabel('Mean IDE', fontsize=13)
ax.set_title('Mean IDE by Dataset and Task', fontsize=14)

ylim = ax.get_ylim()
y_sec = ylim[0] - (ylim[1] - ylim[0]) * 0.22
ax.text(0.5,  y_sec, 'Rest/Movie', ha='center', clip_on=False)
ax.text(hbn_x_start + (len(order_hbn)-1) / 2,  y_sec, 'HBN',    ha='center', clip_on=False)

sns.despine()
plt.tight_layout()
# plt.savefig(f'{PLOT_DIR}/infant_adult_hbn_rest_movie_ide_comparison.pdf', format='pdf', transparent=True)

In [ ]:
model = ols('score ~ C(AgeGroup) * C(task) ', data=subject_means).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print(model.summary())
print(anova_table)


In [ ]:
# HBN: demographics — age distribution by sex and site
bins = np.arange(np.floor(par_df_hbn['age'].min()), np.ceil(par_df_hbn['age'].max()) + 1)
bin_centers = (bins[:-1] + bins[1:]) / 2
sexes = ['Male', 'Female', 'Other']
set2 = sns.color_palette('Set2')
palette_sex = {s: set2[i] for i, s in enumerate(sexes)}

ages_m = par_df_hbn[par_df_hbn['sex'] == 'Male']['age'].dropna()
ages_f = par_df_hbn[par_df_hbn['sex'] == 'Female']['age'].dropna()
ages_o = par_df_hbn[~par_df_hbn['sex'].isin(['Male', 'Female'])]['age'].dropna()
counts_m, _ = np.histogram(ages_m, bins=bins)
counts_f, _ = np.histogram(ages_f, bins=bins)
counts_o, _ = np.histogram(ages_o, bins=bins)

plt.figure(figsize=(8, 4))
width = 0.4
plt.bar(bin_centers - width/3, counts_m, width=width, color=palette_sex['Male'], edgecolor='black', label=f"Male (n={len(ages_m)})")
plt.bar(bin_centers, counts_f, width=width, color=palette_sex['Female'], edgecolor='black', label=f"Female (n={len(ages_f)})")
plt.bar(bin_centers + width/3, counts_o, width=width, color=palette_sex['Other'], edgecolor='black', label=f"Other (n={len(ages_o)})")
plt.xlabel('Age (years)'); plt.ylabel('Count'); plt.title('HBN Age Distribution by Sex')
plt.legend(); sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
# HBN: motion — FD vs Age for movie and rest
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for i, (metric, title) in enumerate([('movie_FD', 'Movie FD vs Age'), ('rest_FD', 'Rest FD vs Age')]):
    ax = axes[i]
    mask = par_df_hbn[['age', metric]].notnull().all(axis=1)
    x, y = par_df_hbn.loc[mask, 'age'], par_df_hbn.loc[mask, metric]
    sns.regplot(data=par_df_hbn.loc[mask], x='age', y=metric, ax=ax,
                scatter_kws={'s': 20, 'alpha': 0.2}, ci=None)
    r_val, p_val = stats.pearsonr(x, y)
    ax.text(0.98, 0.98, f'n={mask.sum()}\nr={r_val:.3f}\np={p_val:.2e}',
            transform=ax.transAxes, ha='right', va='top',
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    ax.set_title(title); ax.set_xlabel('Age (years)'); ax.set_ylabel('Mean FD' if i == 0 else '')
sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
# # HBN: build difference_df (ISC, movie IDE, rest IDE, RestMovieDiff per subject x region)
# import parcelwise_regressions as pr
# participant_df_hbn = pd.read_csv(f'HBN/basic_cohort_info.csv')
# # rename subject col
# temp = results_df_hbn[results_df_hbn['measure']=='ISC']

# dataframe_hbn = pr.join_results_participant_info(results_df_hbn, participant_df_hbn, 'hbn')
# dataframe_hbn = dataframe_hbn[dataframe_hbn['measure']=='TPHATE_DiffOp_IDE']
# # Rename
# dataframe_hbn = dataframe_hbn.rename({"sex_x":'sex', '_age_raw':'Age'}, axis=1)
# # Join dataframe for resut
# difference_df = pr.clean_difference_dataframe(dataframe_hbn)
# # Add back in the ISC Column 
# difference_df['ISC'] = difference_df.merge(temp, on=['subject_id', 'region_name'])['score']
# difference_df.to_csv('HBN/parcelwise_difference_ISC_IDE.csv')


In [ ]:
difference_df = pd.read_csv('HBN/parcelwise_difference_ISC_IDE.csv')

In [ ]:
# HBN: load diff_df CSV (parcelwise difference results)
#diff_df_hbn = pd.read_csv(f'{hu.get_results_dir()}/parcelwise_difference_ISC_IDE.csv')

def get_age_groupings_hbn(age):
    if age < 10: return 'U10'
    if age < 13: return '10-13'
    if age < 17: return '14-17'
    return 'O17'

difference_df['AgeGroup'] = [get_age_groupings_hbn(a) for a in difference_df['Age']]

difference_df_long = pd.melt(
    difference_df, id_vars=['subject_id', 'Age', 'AgeGroup', 'region_name'],
    value_vars=['ISC', 'movie_score', 'rest_score', 'RestMovieDiff'],
    var_name='measure', value_name='score')


In [ ]:
# HBN: surface grid by AgeGroup (ISC, movie IDE, rest IDE, RestMovieDiff)
# age_order_hbn = ['U10', '10-13', '14-17', 'O17']
# measures_hbn_surf = ['ISC', 'movie_score', 'rest_score', 'RestMovieDiff']
# ranges_hbn = [[0.0, 0.3], [0, 18], [0, 18], [-9, 9]]
# cmaps_hbn = [ISC_CMAP, ID_CMAP, ID_CMAP, helper.diverging_colormap_bp()]
# REGION_ORDER_HBN = get_region_order()  # Define this function to return the correct region order for HBN
# for i, measure in enumerate(measures_hbn_surf):
#     sub_df_hbn = difference_df_long[difference_df_long['measure'] == measure].copy()
#     data_arrs_hbn, titles_hbn, img_fns_hbn = [], [], []
#     for age in age_order_hbn:
#         temp = sub_df_hbn[sub_df_hbn['AgeGroup'] == age].groupby('region_name')['score'].mean().reset_index()
#         s = temp.set_index('region_name').reindex(REGION_ORDER_HBN)['score']
#         data_arrs_hbn.append(s)
#         titles_hbn.append(f"{age}")
#         img_fns_hbn.append(f'HBN/plots/{measure}_{age}.png')
    
#     helper.compile_surface_plots_to_grid_by_rows(
#         img_fns_hbn, data_arrs_hbn,
#         output_path=f'{PLOT_DIR}/hbn_{measure.lower()}_agegroup_surface_grid.pdf',
#         n_rows=1, surf_type='fslr', target_density='32k', method='linear',
#         atlas='Schaefer', rerun=True,
#         cmaps_per_row=[cmaps_hbn[i]],
#         cbar_ranges_per_row=[ranges_hbn[i]],
#         cbar_labels_per_row=[measure],
#         titles=titles_hbn
#     )

# Now run the difference maps
measure = 'RestMovieDiff'
this_range = [-9,9]
cmap = helper.diverging_colormap_bp()
sub_df_hbn = difference_df_long[difference_df_long['measure'] == measure].copy()
data_arrs_hbn, titles_hbn, img_fns_hbn = [], [], []
for age in age_order_hbn:
    temp = sub_df_hbn[sub_df_hbn['AgeGroup'] == age].groupby('region_name')['score'].mean().reset_index()
    # Reorger regions
    temp_df = sub_df_hbn[sub_df_hbn['AgeGroup'] == age]
    temp_df = temp_df.set_index(['subject_id', 'region_name']).reindex(REGION_ORDER_HBN, level='region_name').reset_index()
    # Reshape to get [n_subjects x n_regions] array for resampling
    data_arr = np.empty((temp_df['subject_id'].nunique(), len(REGION_ORDER_HBN)))
    for i, subject in enumerate(temp_df['subject_id'].unique()):
        subject_data = temp_df[temp_df['subject_id'] == subject].set_index('region_name').reindex(REGION_ORDER_HBN)['score']
        data_arr[i, :] = subject_data.values

    print(f"Running randomization test for {age} with data shape {data_arr.shape}")
    mean_diff, pvals,reject = stats_helpers.paired_difference_null_distribution(data_arr, arr2=None, alpha=0.01)
    to_plot = np.where(reject, mean_diff, np.nan)  # Use mean_diff for significant regions, NaN otherwise
    print(f'how many reject? {reject.sum()} out of {len(reject)} for age group {age}; nan count in to_plot: {np.isnan(to_plot).sum()}')
    data_arrs_hbn.append(to_plot)
    titles_hbn.append(f"{age}")
    img_fns_hbn.append(f'HBN/plots/{measure}_{age}_thresholded.png')

helper.compile_surface_plots_to_grid_by_rows(
    img_fns_hbn, data_arrs_hbn,
    output_path=f'{PLOT_DIR}/hbn_{measure.lower()}_agegroup_surface_grid_thresholded.pdf',
    n_rows=1, surf_type='fslr', target_density='32k', method='linear',
    atlas='Schaefer', rerun=True,
    cmaps_per_row=[cmap],
    cbar_ranges_per_row=[this_range],
    cbar_labels_per_row=[measure],
    titles=titles_hbn
)

In [ ]:
# HBN: average barplots (rest IDE, movie IDE, RestMovieDiff, ISC by AgeGroup)
df1_hbn = difference_df.groupby(['subject_id', 'AgeGroup']).mean(numeric_only=True).reset_index()
fig, ax = plt.subplots(2, 2, figsize=(16, 8), sharex=True)
ax = ax.ravel()
age_order_orig = sorted(df1_hbn['AgeGroup'].unique())
for i, (col, title, ylim) in enumerate([
    ('rest_score', 'Rest IDE by Age Group', (0, 12)),
    ('movie_score', 'Movie IDE by Age Group', (0, 12)),
    ('RestMovieDiff', 'Rest - Movie IDE by Age Group', (0, 5)),
    ('ISC', 'ISC by Age Group', (0, 0.3))
]):
    g = sns.barplot(x='AgeGroup', y=col, data=df1_hbn, palette='magma', alpha=0.7,
                    order=age_order_orig, ax=ax[i])
    g.set(title=title, ylim=ylim, ylabel=col if i % 2 == 0 else '')
ax[2].tick_params(axis='x', rotation=45); ax[3].tick_params(axis='x', rotation=45)
sns.despine(); plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/hbn_avg_isc_ide_restmovediff_barplots.pdf', format='pdf', transparent=True)
plt.show()

In [ ]:
# HBN: parcelwise regressions — ISC ~ Age, movie IDE ~ Age, rest IDE ~ Age, RestMovieDiff ~ Age
temp1_hbn = difference_df.copy()
temp1_hbn['log_age'] = np.log(temp1_hbn['Age'] + 1)
regression_specs = [
    ("ISC", "ISC ~ Age + sex + session + movie_FD", ['movie_FD', 'ISC', 'sex', 'session', 'Age'],
     'hbn_isc_age_coef_surface.pdf', ['Age']),
    ("movie_score", "movie_score ~ Age + ISC + sex + session + movie_FD", ['ISC', 'movie_FD', 'sex', 'session', 'Age'],
     'hbn_movie_ide_age_coef_surface.pdf', ['Age', 'ISC']),
    ("rest_score", "rest_score ~ Age + sex + session + rest_FD", ['rest_FD', 'sex', 'session', 'Age'],
     'hbn_rest_ide_age_coef_surface.pdf', ['Age']),
    ("RestMovieDiff", "RestMovieDiff ~ Age + ISC + sex + session + movie_FD + rest_FD",
     ['movie_FD', 'ISC', 'rest_FD', 'sex', 'session', 'Age'],
     'hbn_restmovediff_age_coef_surface.pdf', ['Age', 'ISC']),
     ("RestMovieDiff", "RestMovieDiff ~ log_age + ISC + sex + session + movie_FD + rest_FD",
     ['movie_FD', 'ISC', 'rest_FD', 'sex', 'session', 'log_age'],
     'hbn_restmovediff_log_age_coef_surface.pdf', ['log_age', 'ISC']),
]

for yname, formula, covariates, outfn, to_plot in regression_specs:
    print(f"Running regression: {formula}")
    results_reg = pwr.run_regression_analyses(
        temp1_hbn, 'Age', yname, formula, covariates,
        output_directory=None, root_filename=None, region_order=REGION_ORDER_HBN,
        title='', plot=0, verbose=0)
    for var in to_plot:
        sig_mask = results_reg.set_index('region_name').reindex(REGION_ORDER_HBN)[f'sig_{var}_fdr'].values
        coef_masked = results_reg.set_index('region_name').reindex(REGION_ORDER_HBN)[f'coef_{var}'].values * sig_mask
        cbar_range, this_cmap = helper.determine_colorbar_range(coef_masked, helper.diverging_colormap_gpu(), symmetric=True)
        fn = outfn.replace('coef_surface', f'{var.lower()}_coef_surface')
        helper.generate_surface_plot(
            coef_masked, image_fn=f'{PLOT_DIR}/{fn}',
            atlas='Schaefer', cmap=this_cmap, cbar_range=cbar_range,
            surf_type='fslr', target_density='32k', method='linear',
            include_cbar=True, title=rf'$\beta$_{var} ({yname})', threshold=None, mask_medial_wall=True
        )

### Fix the below two with the saved data

In [ ]:
df_corr_hbn = pd.read_csv('compiled/results/combined_corrs.csv')
df_corr_hbn = df_corr_hbn[df_corr_hbn['dataset']=='HBN'].reset_index(drop=True)
print(f"ISC-IDE correlation computed for {len(df_corr_hbn)} HBN subjects")

In [ ]:
# HBN: ISC-IDE barplot by fine AgeGroup + scatter vs Age
def get_age_groupings_hbn_fine(age):
    if age < 8: return 'U08'
    if age < 9: return 'U09'
    if age < 10: return 'U10'
    if age < 11: return 'U11'
    if age < 12: return 'U12'
    if age < 13: return 'U13'
    if age < 14: return 'U14'
    if age < 15: return 'U15'
    if age < 16: return 'U16'
    if age < 17: return 'U17'
    return 'U22'

df_corr_hbn['AgeGroup2'] = [get_age_groupings_hbn_fine(a) for a in df_corr_hbn['Age'].astype(float)]
age_order_hbn_fine = ['U08', 'U09', 'U10', 'U11', 'U12', 'U13', 'U14', 'U15', 'U16', 'U17', 'U22']
age_group_counts_hbn = df_corr_hbn['AgeGroup2'].value_counts().to_dict()

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x='AgeGroup2', y='zscore', data=df_corr_hbn, palette='magma', alpha=0.7,
            order=age_order_hbn_fine, ax=ax, edgecolor='black', ci=None)
sns.stripplot(x='AgeGroup2', y='zscore', data=df_corr_hbn, palette='magma', size=5, alpha=1,
              order=age_order_hbn_fine, ax=ax)
for p, age_group in zip(ax.patches, age_order_hbn_fine):
    count = age_group_counts_hbn.get(age_group, 0)
    ax.text(p.get_x() + p.get_width() / 2., 0.01, f'n={count}', ha="center", va="bottom", fontsize=8)
ax.set_xlabel('Age Group'); ax.set_ylabel('ISC-IDE correlation (z-score)')
ax.set_title('Subject-wise ISC vs IDE (movie) correlation by Age Group')
sns.despine(); plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/hbn_isc_ide_corr_by_agegroup.pdf', format='pdf', transparent=True)
plt.show()

# Scatter vs Age
plt.figure(figsize=(8, 6))
sns.regplot(x='Age', y='zscore', data=df_corr_hbn, scatter_kws={'s': 20, 'alpha': 0.5})
rho, p = scipy.stats.spearmanr(df_corr_hbn['Age'].astype(float), df_corr_hbn['zscore'].astype(float), nan_policy='omit')
plt.gca().text(0.95, 0.05, f'rho={rho:.3f}\np={p:.3g}\nN={len(df_corr_hbn)}',
               transform=plt.gca().transAxes, ha='right', va='bottom',
               bbox=dict(facecolor='white', alpha=0.7))
plt.title('Subject-wise ISC vs IDE (movie) correlation vs Age')
plt.xlabel('Age'); plt.ylabel('ISC-IDE z-score')
sns.despine(); plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/hbn_isc_ide_corr_vs_age_scatter.pdf', format='pdf', transparent=True)
plt.show()

# Section 5: Infant Rest/Movie

In [ ]:
# Infant: load average results for aeronaut and sleep
results_dir_inf = 'infant_restmovie/results'
infant_tasks = ['aeronaut', 'sleep']
measures_inf = {'ISC': 'ISC', 'TPHATE_DiffOp_IDE': 'IDE'}

avg_results_inf = {}
all_results_inf = {}
avg_imgs_inf = {}
for task in infant_tasks:
    avg_results_inf[task] = {}
    all_results_inf[task] = {}
    avg_imgs_inf[task] = {}
    mask_file = f'infant_restmovie/masks/InfantRestMovie_intersect_mask_{task}.nii.gz'
    masker_inf = NiftiMasker(mask_img=mask_file)
    for measure in measures_inf.keys():
        fn = f'{results_dir_inf}/{task}_{measure}_all_subjects_results.nii.gz'
        if os.path.exists(fn):
            img = nib.load(fn)
            masked = masker_inf.fit_transform(img)
            all_results_inf[task][measure] = masked
            mean_data = np.nanmean(img.get_fdata(), axis=3)
            avg_imgs_inf[task][measure] = nib.Nifti1Image(mean_data, img.affine, img.header)
            avg_results_inf[task][measure] = masked
            print(f"Loaded infant {task} {measure}: {masked.shape}, mean={np.nanmean(masked):.3f}, std={np.mean(np.nanstd(masked, axis=1)):.3f}")
        else:
            print(f"Warning: {fn} not found") 


In [ ]:
# ide_subject_averages_arm = []
# isc_subject_averages_arm = []
# mask = nib.load(f'adult_restmovie/masks/adult_restmovie_intersect_mask.nii.gz')
# masker = NiftiMasker(mask_img=mask)
# for task in ['aeronaut', 'rest']:
#     ide_fn = f'{results_dir_arm}/{task}_TPHATE_DiffOp_IDE_all_subjects_results.nii.gz'
#     isc_fn = f'{results_dir_arm}/{task}_ISC_all_subjects_results.nii.gz'
#     for target_list, fn, measure in zip([ide_subject_averages_arm, isc_subject_averages_arm], [ide_fn, isc_fn], ['IDE','ISC']):
#         if os.path.exists(fn):
#             img = nib.load(fn)
#             data = masker.fit_transform(img)
#             n_subjects = data.shape[0]
#             for subj_idx in range(n_subjects):
#                 mean_val = np.nanmean(data[subj_idx])
#                 std_val = np.nanstd(data[subj_idx])
#                 target_list.append({'task': task, 'subject_id': subj_idx, f'mean_{measure}': mean_val, f'std_{measure}': std_val})
#             print(f"Loaded {task} {measure}: {n_subjects} subjects, mean across subjects = {np.nanmean(data):.3f}, sd = {np.mean(np.nanstd(data, axis=1)):.3f}")
#         else:            
#             print(f"Warning: {fn} not found")
    

# ide_df_arm = pd.DataFrame(ide_subject_averages_arm)
# isc_df_arm = pd.DataFrame(isc_subject_averages_arm)


arr_id = all_results_inf['aeronaut']['TPHATE_DiffOp_IDE']
print(arr_id.shape, ide_df_arm[ide_df_arm['task'] == 'aeronaut']['mean_IDE'].values.shape)
print('IDE Results: ',ttest_ind(np.mean(arr_id, axis=1), ide_df_arm[ide_df_arm['task'] == 'aeronaut']['mean_IDE'].values, equal_var=False))

# do the same for ISC
arr_isc = all_results_inf['aeronaut']['ISC']
print(arr_isc.shape, isc_df_arm[isc_df_arm['task'] == 'aeronaut']['mean_ISC'].values.shape)
print('ISC Results: ',ttest_ind(np.mean(arr_isc, axis=1), isc_df_arm[isc_df_arm['task'] == 'aeronaut']['mean_ISC'].values,  equal_var=False))


In [ ]:
# Infant: surface grid (aeronaut ISC, aeronaut IDE, sleep IDE)
nifti_imgs_inf = [
    avg_imgs_inf['aeronaut']['ISC'],
    avg_imgs_inf['aeronaut']['TPHATE_DiffOp_IDE'],
    avg_imgs_inf['sleep']['TPHATE_DiffOp_IDE'],
]
titles_inf = ['Aeronaut ISC', 'Aeronaut IDE', 'Sleep IDE']
cbar_ranges_inf = [(0, 0.4), (1, 18), (1, 25)]
cmaps_inf = [ISC_CMAP, ID_CMAP, ID_CMAP]

temp_fns_inf = []
for idx, (img, title, cmap, cbar_range) in enumerate(zip(nifti_imgs_inf, titles_inf, cmaps_inf, cbar_ranges_inf)):
    fn = title.lower().replace(' ', '_')
    temp_fn = f'compiled/plots/infant_restmovie_{fn}_surface.png'
    helper.generate_surface_plot(
        data_fn=img, image_fn=temp_fn, atlas='searchlight', cmap=cmap,
        cbar_range=cbar_range, surf_type='fslr', target_density='32k',
        include_cbar=True, title=title, method='linear', threshold=None, mask_medial_wall=True
    )



In [ ]:
# Infant: TFCE-thresholded aeronaut-sleep difference map
d1_inf = nib.load('infant_restmovie/results/sleep_aeronaut_two_samp_ttest_output_tfce_corrp_tstat1.nii.gz')
mask_inf_diff = image.math_img('np.where(X >= 0.95, 1, 0)', X=d1_inf)
mean_img_inf = nib.load('infant_restmovie/results/aeronaut_sleep_TPHATE_DiffOp_IDE_SL_difference_maps_avg_unthresholded.nii.gz')
masked_mean_img_inf = image.math_img('X * Y', X=mean_img_inf, Y=mask_inf_diff)

helper.generate_surface_plot(
    data_fn=masked_mean_img_inf, image_fn=None,
    atlas='searchlight', surf_type='fslr', target_density='32k', method='linear',
    cmap=helper.diverging_colormap_bp(), cbar_range=(-11, 11),
    include_cbar=True, title='Aeronaut - Sleep IDE (TFCE thresholded)', mask_medial_wall=True
)

In [ ]:
# Infant: composite plot — diff map + top ISC overlay
isc_img_inf = avg_imgs_inf['aeronaut']['ISC']
diff_img_inf = masked_mean_img_inf

isc_data_inf = isc_img_inf.get_fdata()
isc_valid_inf = isc_data_inf[~np.isnan(isc_data_inf)]
threshold_inf = 95
isc_thr_inf = np.nanpercentile(isc_valid_inf, threshold_inf)
top_mask_inf = (isc_data_inf >= isc_thr_inf) & ~np.isnan(isc_data_inf)

top_mask_vol_inf = np.full(diff_img_inf.shape, np.nan, dtype=float)
top_mask_vol_inf[top_mask_inf] = 1.0
top_mask_img_inf = nib.Nifti1Image(top_mask_vol_inf, affine=diff_img_inf.affine, header=diff_img_inf.header)

helper.generate_surface_plot(
    data_fn=[diff_img_inf, top_mask_img_inf],
    image_fn=f'{PLOT_DIR}/infant_aeronaut_sleep_diff_top_isc_overlay.pdf',
    atlas='searchlight', cmap=helper.diverging_colormap_bp(), cbar_range=(-11, 11),
    surf_type='fslr', target_density='32k', include_cbar=True,
    title='Aeronaut − Sleep IDE (thresholded) + Top ISC voxels',
    method='linear', threshold=None, alpha=0.5, mask_medial_wall=True,
    layers_kwargs=[
        dict(cmap=helper.diverging_colormap_bp(), cbar_range=(-11, 11), alpha=1,
             label='ΔIDE (Aeronaut − Sleep)', cbar=True),
        dict(cmap=ListedColormap([(1, 1, 0.6, 1.0)]), cbar_range=(0, 1), alpha=0.8,
             label=f'Top {100 - threshold_inf}% ISC mask', cbar=True),
    ],
)

In [ ]:
# # Infant: ISC-IDE permutation correlation per subject (aeronaut, parcelwise)
# par_df_inf = pd.read_csv('infant_restmovie/participant_info.csv')
# results_df_inf = pd.read_csv('infant_restmovie/results/parcelwise_results_ISC_IDE.csv', index_col=0)
# par_df_inf_aero = par_df_inf[par_df_inf['task'] == 'aeronaut'].reset_index(drop=True)
# results_df_inf_aero = results_df_inf[results_df_inf['task'] == 'aeronaut']

# df_corr_inf = pd.DataFrame(columns=['subject', 'rho', 'pval', 'zscore', 'age'])
# for s in results_df_inf_aero['subject'].unique():
#     temp = results_df_inf_aero[results_df_inf_aero['subject'] == s]
#     if len(temp) < 400:
#         continue
#     ide_vals = temp[temp['measure'] == 'TPHATE_DiffOp_IDE']['score'].values
#     isc_vals = temp[temp['measure'] == 'ISC']['score'].values
#     pval, obs, zsc = sh.permute_pattern(ide_vals, isc_vals, n_permutations=1000, random_state=None)
#     age_val = par_df_inf_aero[par_df_inf_aero['subject_id'] == s]['age'].values
#     age_val = age_val[0] if len(age_val) > 0 else np.nan
#     df_corr_inf.loc[len(df_corr_inf)] = [s, obs, pval, zsc, age_val]
# df_corr_inf.to_csv('infant_restmovie/results/isc_ide_correlation_null_stats.csv', index=False)
df_corr_inf = pd.read_csv('compiled/results/combined_corr_analyses.csv')
df_corr_inf = df_corr_inf[df_corr_inf['dataset'] == 'infant_restmovie']
print(f"Infant ISC-IDE correlation computed for {len(df_corr_inf)} subjects")

In [ ]:
# Infant: mean IDE barplot by task (aeronaut, mickey, sleep)
ide_subject_avgs_inf = []
for task in ['aeronaut', 'mickey', 'sleep']:
    ide_fn = f'{results_dir_inf}/{task}_TPHATE_DiffOp_IDE_all_subjects_results.nii.gz'
    if os.path.exists(ide_fn):
        ide_img = nib.load(ide_fn)
        ide_data = ide_img.get_fdata()
        n_subjects = ide_data.shape[3]
        for subj_idx in range(n_subjects):
            mean_ide = np.nanmean(ide_data[:, :, :, subj_idx])
            ide_subject_avgs_inf.append({'task': task, 'subject_id': subj_idx, 'mean_IDE': mean_ide})
        print(f"Loaded infant {task}: {n_subjects} subjects")
    else:
        print(f"Warning: {ide_fn} not found")

ide_df_inf = pd.DataFrame(ide_subject_avgs_inf)
tasks_inf = ['aeronaut', 'mickey', 'sleep']
color=helper.dataset_colors('infant_restmovie')
plt.figure(figsize=(4,4))
sns.barplot(data=ide_df_inf, x='task', y='mean_IDE', color=color,
            edgecolor='k', linewidth=2, alpha=0.8)
sns.stripplot(data=ide_df_inf, x='task', y='mean_IDE', color=color, 
              size=12, alpha=1, jitter=True, edgecolor='k', linewidth=1)
y_max_inf = ide_df_inf['mean_IDE'].max()
for idx, (t1, t2) in enumerate([('aeronaut', 'mickey'), ('aeronaut', 'sleep'), ('mickey', 'sleep')]):
    d1 = ide_df_inf[ide_df_inf['task'] == t1]['mean_IDE'].values
    d2 = ide_df_inf[ide_df_inf['task'] == t2]['mean_IDE'].values
    _, p_val = ttest_ind(d1, d2)
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'n.s.'
    y = y_max_inf + 0.2 + idx * 0.4
    x1, x2 = tasks_inf.index(t1), tasks_inf.index(t2)
    plt.plot([x1, x2], [y, y], lw=1.5, c='k')
    plt.text((x1 + x2) * .5, y + 0.05, sig, ha='center', va='bottom', color='k', fontsize=14)
plt.ylabel('Mean IDE (across voxels)', fontsize=14)
plt.xlabel('Task', fontsize=14)
plt.title('Infant Average IDE by Task', fontsize=16)
sns.despine(); plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/infant_avg_ide_by_task_barplot.pdf', format='pdf', transparent=True)
plt.show()

In [ ]:
# Infant: parcelwise regression TPHATE_DiffOp_IDE ~ ISC + age (aeronaut) 
results_df_inf = pd.read_csv('compiled/results/infant_restmovie_compiled_results.csv',index_col=0) 
results_df_inf_aero = results_df_inf[results_df_inf['task'] == 'aeronaut'].copy().reset_index()
par_df = pd.read_csv('compiled/info/combined_participant_info.csv')
par_df_inf_aero = par_df[(par_df['dataset'] == 'InfantRestMovie') & (par_df['task'] == 'aeronaut')].reset_index(drop=True)
REGION_ORDER_INF = get_region_order()   
results_df_inf1 = results_df_inf_aero.pivot_table(
    index=['subject_id', 'region_name', 'task'], columns='measure', values='score').reset_index()
results_df_inf1['age_months'] = [
    par_df_inf_aero[par_df_inf_aero['participant_id'] == s]['age_months'].values[0]
    if len(par_df_inf_aero[par_df_inf_aero['participant_id'] == s]) > 0 else np.nan
    for s in results_df_inf1['subject_id'].values]
# log age 
results_df_inf1['log_age'] = np.log(results_df_inf1['age_months']+1)
# add in sex and FD
results_df_inf1['sex'] = [
    par_df_inf_aero[par_df_inf_aero['participant_id'] == s]['sex'].values[0]
    if len(par_df_inf_aero[par_df_inf_aero['participant_id'] == s]) > 0 else np.nan
    for s in results_df_inf1['subject_id'].values]
results_df_inf1['movie_FD'] = [
    par_df_inf_aero[par_df_inf_aero['participant_id'] == s]['movie_FD'].values[0]
    if len(par_df_inf_aero[par_df_inf_aero['participant_id'] == s]) > 0 else np.nan
    for s in results_df_inf1['subject_id'].values]

formula_inf = "TPHATE_DiffOp_IDE ~ ISC + log_age + sex + movie_FD"
a_inf = stats_helpers.parcelwise_regression(results_df_inf1, "log_age", yname="TPHATE_DiffOp_IDE",
                                             formula=formula_inf, region_order=REGION_ORDER_INF,
                                             alpha=0.05, fdr_method='fdr_bh')
a_inf = a_inf.set_index('region_name').reindex(REGION_ORDER_INF).reset_index()

for var, label in [('log_age', 'log_age'), ('ISC', 'isc')]:
    sig_mask = a_inf[f'sig_{var}_fdr'].values
    print(f"For variable '{var}', number of significant regions after FDR correction: {sig_mask.sum()} out of {len(sig_mask)}")
    coef_masked = a_inf[f'coef_{var}'].values * sig_mask
    cbar_range, this_cmap = helper.determine_colorbar_range(coef_masked, helper.diverging_colormap_gpu(), symmetric=True)
    helper.generate_surface_plot(
        coef_masked,
        image_fn=f'{PLOT_DIR}/infant_log_age_predicts_ide_{label}_coef_surface.pdf',
        atlas='Schaefer', cmap=this_cmap, cbar_range=cbar_range,
        surf_type='fslr', target_density='32k', method='linear',
        include_cbar=True, title=rf'$\beta$_{var} (infant aeronaut IDE)', threshold=None, mask_medial_wall=True
    )

# Section 6: Comparing ID collapse and ISC across datasets

In [ ]:
hbn_difference_df = pd.read_csv('HBN/parcelwise_difference_ISC_IDE.csv')
hbn_difference_df.head()

In [ ]:
 # Cross-dataset: combine delta_ID (rest − movie IDE) and ISC by parcel and subject
# Adult Rest/Movie: tasks are aeronaut (movie) and rest
# HBN:             tasks are movieTP and rest; difference_df already built in cell 42

# === Adult Rest/Movie ===
arm_csv = pd.read_csv('compiled/results/adult_restmovie_compiled_results.csv')

# Pivot IDE scores by task → compute delta_ID = rest − aeronaut
arm_ide = arm_csv[arm_csv['measure'] == 'TPHATE_DiffOp_IDE'].copy()
arm_ide_pivot = arm_ide.pivot_table(
    index=['subject_id', 'region_name'], columns='task', values='score'
).reset_index()
arm_ide_pivot = arm_ide_pivot.dropna(subset=['aeronaut', 'rest'])
arm_ide_pivot['delta_ID'] = arm_ide_pivot['rest'] - arm_ide_pivot['aeronaut']

# Join aeronaut ISC (movie-viewing ISC is the natural pair for delta_ID)
arm_isc = (arm_csv[(arm_csv['measure'] == 'ISC') & (arm_csv['task'] == 'aeronaut')]
        [['subject_id', 'region_name', 'score']]
        .rename(columns={'score': 'ISC'}))
arm_pivot = arm_ide_pivot.merge(arm_isc, on=['subject_id', 'region_name'], how='inner')

# Add age
arm_age = arm_csv[['subject_id', '_age_raw']].drop_duplicates()
arm_pivot = arm_pivot.merge(arm_age, on='subject_id', how='left').rename(columns={'_age_raw':
'age'})
arm_pivot['dataset'] = 'adult_restmovie'
arm_pivot.to_csv('adult_restmovie/parcelwise_difference_ISC_IDE.csv', index=False)
# === HBN ===
# difference_df already has RestMovieDiff (rest − movie), ISC, and Age from cell 42
hbn_pivot = hbn_difference_df[['subject_id', 'region_name', 'RestMovieDiff', 'ISC', 'Age']].copy()
hbn_pivot = hbn_pivot.rename(columns={'RestMovieDiff': 'delta_ID', 'Age': 'age'})
hbn_pivot['dataset'] = 'HBN'

# === Combine ===
keep_cols = ['subject_id', 'region_name', 'delta_ID', 'ISC', 'age', 'dataset']
combined_delta_isc = pd.concat(
    [arm_pivot[keep_cols], hbn_pivot[keep_cols]],
    ignore_index=True
)

combined_delta_isc.to_csv('compiled/results/combined_delta_id_isc.csv', index=False)
print(f"Combined: {combined_delta_isc.shape[0]} rows across {combined_delta_isc['dataset'].nunique()} datasets")
print(combined_delta_isc.groupby('dataset')[['delta_ID', 'ISC']].describe())
combined_delta_isc.head()

In [ ]:
combined_delta_isc = pd.read_csv('compiled/results/combined_delta_id_isc.csv')
combined_delta_isc['age_months'] = combined_delta_isc['age'] * 12
combined_delta_isc['age_months_z'] = stats.zscore(combined_delta_isc['age_months'])
# normalize delta_ID and ISC within each subject to put on same scale
combined_delta_isc['delta_ID_z'] = combined_delta_isc.groupby('subject_id')['delta_ID'].transform(lambda x: stats.zscore(x, nan_policy='omit'))
combined_delta_isc['ISC_z'] = combined_delta_isc.groupby('subject_id')['ISC'].transform(lambda x: stats.zscore(x, nan_policy='omit'))

In [ ]:
# res = stats_helpers.within_subject_spearman(combined_delta_isc, 'ISC_z', 'delta_ID_z', subject_col='subject_id')
participant_df = pd.read_csv('compiled/info/combined_participant_info.csv')
participant_df.dataset.unique()

In [ ]:
def group_ages(age):
    if age < 8: return 'U08'
    if age < 10: return '8-10'
    if age < 12: return '10-12'
    if age < 14: return '12-15'
    if age < 18: return '15-17'
    else: return 'O17'


In [ ]:
# res = res[['subject_id', 'rho', 'pval', 'zscore' ]]
# participant_df = pd.read_csv('compiled/info/combined_participant_info.csv')
# participant_df = participant_df[participant_df['dataset'].isin(['HBN', 'AdultRestMovie'])]
# participant_df = participant_df.rename(columns={'participant_id': 'subject_id'})
# res = res.merge(participant_df[['subject_id', 'age_months', 'dataset']], on='subject_id', how='left')
# res['age'] = res['age_months'] / 12
# res['age_group'] = res['age'].apply(group_ages)
# res.to_csv('compiled/results/isc_deltaid_subjectwise_correlation.csv', index=False)
res = pd.read_csv('compiled/results/isc_deltaid_subjectwise_correlation.csv')

## Infant group bootstrapping
Two independent pools of infants (sleep, movie watch). generate a distribution of group-level deltaID maps that reflect the uncertainty of unmatched groups, get a single rho(ISC, deltaID) with confidence interval that exists on the same axis as the subject-level values from HBN / Adults
bootstrapping group means rather than IDs -- each iteration resamples sleepers & computes mean ID map, resamples movies & computes mean ID map, subtracts, correlates with group ISC map. 


In [ ]:
# Infant dataset: group-mean resampling to generate null distribution for aeronaut-sleep IDE difference map
inf_df = pd.read_csv('compiled/results/infant_restmovie_compiled_results.csv',index_col=0)
inf_ages = pd.read_csv('compiled/info/combined_participant_info.csv')
inf_ages = inf_ages[inf_ages['dataset'] == 'InfantRestMovie']['age_months'].values
inf_df.head()

# Load in the summary dataframe and the bootstrapped null distribution results
combined_delta_isc_all= pd.read_csv('compiled/results/combined_delta_id_isc_all_datasets.csv')
summary_df = pd.read_csv('compiled/results/isc_deltaid_developmental_summary.csv')
z_boot = np.load('compiled/results/infant_restmovie_isc_deltaid_z_bootstrap_distribution.npy')
res = pd.read_csv('compiled/results/isc_deltaid_subjectwise_correlation.csv')


### all of these generate the data loaded above

In [ ]:
def bootstrap_rho_isc_deltaID_unmatched(id_sleep, id_movie, isc_movie, n_iterations=10000, seed=4, zscore_within_iteration=True):
    """
    Parameters
    ----------
    id_sleep  : (n_sleepers, n_parcels)
    id_movie  : (n_movie,    n_parcels)
    isc_movie : (n_parcels,)  -- group-mean ISC, treated as fixed
    n_iterations : int
    zscore_within_iteration : bool
        If True, z-score deltaID and ISC across parcels within each bootstrap
        iteration before computing rho. This matches the within-subject z-scoring
        done for matched participants, making the infant rho directly comparable.
        If False, raw Spearman rho is computed (still rank-based, so comparable,
        but slightly different in the presence of outlier parcels).

    Returns
    -------
    rho_observed  : float  -- point estimate from full group means (no resampling)
    zscore_observed : float  -- (rho_observed - mean(rho_boot)) / std(rho_boot),
                               matching the z-score convention in within_subject_spearman
    rho_boot      : (n_iterations,) array of bootstrap rho values
    ci_lower, ci_upper       : float  -- 2.5th/97.5th percentile of rho_boot
    ci_lower_z, ci_upper_z   : float  -- CI bounds converted to the same z-score scale
    """

    rng_boot = np.random.default_rng(seed)
    n_sleepers, n_parcels = id_sleep.shape
    n_movie               = id_movie.shape[0]

    # --- Point estimate from full group means (no resampling) ----------------
    mean_sleep_obs = id_sleep.mean(axis=0)
    mean_movie_obs = id_movie.mean(axis=0)
    delta_obs      = mean_sleep_obs - mean_movie_obs

    if zscore_within_iteration:
        delta_obs_z = stats.zscore(delta_obs)
        isc_z       = stats.zscore(isc_movie)
    else:
        delta_obs_z = delta_obs
        isc_z       = isc_movie

    _, rho_observed, z_observed = stats_helpers.permute_pattern(delta_obs_z, isc_z,
            n_permutations=n_iterations, random_state=seed
        )

    # --- Bootstrap -----------------------------------------------------------
    rho_boot = np.empty(n_iterations)
    z_boot = np.empty(n_iterations)
    for i in range(n_iterations):
        idx_sleep = rng_boot.integers(0, n_sleepers, size=n_sleepers)
        idx_movie = rng_boot.integers(0, n_movie,    size=n_movie)

        mean_sleep_b = id_sleep[idx_sleep].mean(axis=0)
        mean_movie_b = id_movie[idx_movie].mean(axis=0)
        delta_b      = mean_sleep_b - mean_movie_b

        if zscore_within_iteration:
            delta_b_z = stats.zscore(delta_b)
        else:
            delta_b_z = delta_b
        _, rboot, zboot = stats_helpers.permute_pattern(delta_b_z, isc_z,
            n_permutations=n_iterations, random_state=seed
        )
        rho_boot[i] = rboot
        z_boot[i]   = zboot
        if i % 100 == 0:
            print(f"Bootstrap iteration {i}/{n_iterations} completed")

    # Compute stats around
    rho_ci_lower = np.percentile(rho_boot, 2.5)
    rho_ci_upper = np.percentile(rho_boot, 97.5)
    rho_boot_mean = np.mean(rho_boot)
    rho_boot_std  = np.std(rho_boot)
    zscore_rho_observed = (rho_observed - rho_boot_mean) / rho_boot_std

    # Repeat for the z-scores
    z_ci_lower = np.percentile(z_boot, 2.5)
    z_ci_upper = np.percentile(z_boot, 97.5)
    z_boot_mean = np.mean(z_boot)
    z_boot_std  = np.std(z_boot)
    zscore_z_observed = (z_observed - z_boot_mean) / z_boot_std
    
    print(f'Rho: observed = {rho_observed:.4f} (95% CI: [{rho_ci_lower:.4f}, {rho_ci_upper:.4f}]), z-score = {zscore_rho_observed:.4f}')
    print(f'Z: observed = {z_observed:.4f} (95% CI: [{z_ci_lower:.4f}, {z_ci_upper:.4f}]), z-score = {zscore_z_observed:.4f})')
    return rho_observed, rho_ci_lower, rho_ci_upper, rho_boot, z_observed, z_ci_lower, z_ci_upper, z_boot


In [ ]:
# get age distributions for infants
infant_info = pd.read_csv('compiled/info/combined_participant_info.csv')
infant_info = infant_info[infant_info['dataset'] == 'InfantRestMovie'].reset_index(drop=True)

# Divide infants into two groups based on median age for group-mean resampling
median_age = infant_info['age_months'].median()
infant_info['age_group'] = infant_info['age_months'].apply(lambda x: 'younger' if x < median_age else 'older')
print(infant_info['age_group'].value_counts())

# now run bootstrapping to get a mean and CI for each quarter of infant ages
bootstrap_results = {}
for age_group in infant_info['age_group'].unique():
    print(f"Running bootstrap for age group {age_group}")
    group_subjects = infant_info[infant_info['age_group'] == age_group]['participant_id'].values
    # drop duplicate
    group_subjects = group_subjects[group_subjects!='s3097_1_4']
    id_sleep = inf_df[(inf_df['task'] == 'sleep') & (inf_df['subject_id'].isin(group_subjects)) & (inf_df['measure'] == 'TPHATE_DiffOp_IDE')].reset_index(drop=True)
    id_movie = inf_df[(inf_df['task'] == 'aeronaut') & (inf_df['subject_id'].isin(group_subjects)) & (inf_df['measure'] == 'TPHATE_DiffOp_IDE')].reset_index(drop=True)
    isc_movie = inf_df[(inf_df['task'] == 'aeronaut') & (inf_df['subject_id'].isin(group_subjects)) & (inf_df['measure'] == 'ISC')].reset_index(drop=True)
    
    # pivot to get subjects x parcels
    id_sleep_pivot = id_sleep.pivot(index='subject_id', columns='region_name', values='score').values
    id_movie_pivot = id_movie.pivot(index='subject_id', columns='region_name', values='score').values
    isc_movie_pivot = isc_movie.groupby('region_name')['score'].mean().values  # group-mean ISC across all subjects in this age group

    results = bootstrap_rho_isc_deltaID_unmatched(id_sleep_pivot, id_movie_pivot, isc_movie_pivot,
        n_iterations=1000, seed=4, zscore_within_iteration=True)
    bootstrap_results[age_group] = results

# Save results to a DataFrame
bootstrap_summary = []
for age_group, (rho_obs, rho_ci_low, rho_ci_up, rho_boot, z_obs, z_ci_low, z_ci_up, z_boot) in bootstrap_results.items():
    bootstrap_summary.append({
        'age_group': age_group,
        'rho_observed': rho_obs,
        'rho_ci_lower': rho_ci_low,
        'rho_ci_upper': rho_ci_up,
        'z_observed': z_obs,
        'z_ci_lower': z_ci_low,
        'z_ci_upper': z_ci_up
    })
bootstrap_summary_df = pd.DataFrame(bootstrap_summary)
bootstrap_summary_df.to_csv('compiled/results/infant_isc_deltaid_bootstrap_summary_median_split.csv', index=False)
# also save the bootstrap distributions for potential future plotting
for age_group, (rho_obs, rho_ci_low, rho_ci_up, rho_boot, z_obs, z_ci_low, z_ci_up, z_boot) in bootstrap_results.items():
    np.save(f'compiled/results/infant_isc_deltaid_bootstrap_rho_{age_group}_median_split.npy', rho_boot)
    np.save(f'compiled/results/infant_isc_deltaid_bootstrap_z_{age_group}_median_split.npy', z_boot)

In [ ]:
id_sleep = inf_df[(inf_df['task'] == 'sleep') & (inf_df['measure'] == 'TPHATE_DiffOp_IDE')].pivot_table(index='subject_id', columns='region_name', values='score').values
id_movie = inf_df[(inf_df['task'] == 'aeronaut') & (inf_df['measure'] == 'TPHATE_DiffOp_IDE')].pivot_table(index='subject_id', columns='region_name', values='score').values
isc_movie = inf_df[(inf_df['task'] == 'aeronaut') & (inf_df['measure'] == 'ISC')].groupby('region_name')['score'].mean().values

# Now z=score ID and ISC within each subject to put on same scale (matching the within-subject z-scoring done for matched participants)
id_sleep = stats.zscore(id_sleep, axis=1, nan_policy='omit')
id_movie = stats.zscore(id_movie, axis=1, nan_policy='omit')
isc_movie = stats.zscore(isc_movie, nan_policy='omit')

print(f"ID_sleep shape: {id_sleep.shape}, ID_movie shape: {id_movie.shape}, ISC_movie shape: {isc_movie.shape}")

rho_observed, rho_ci_lower, rho_ci_upper, rho_boot, z_observed, z_ci_lower, z_ci_upper, z_boot = bootstrap_rho_isc_deltaID_unmatched(
    id_sleep, id_movie, isc_movie, n_iterations=1000,
)

# One-sample test: is the bootstrap distribution different from zero?
# Use the percentile method: if CI excludes 0, reject null.
# Alternatively, use a bootstrap p-value:
p_twotailed = 2 * min(
    np.mean(rho_boot >= 0),
    np.mean(rho_boot <= 0),
)
print(f"  Bootstrap p (two-tailed, H0: ρ=0): {p_twotailed:.4f}")

p_twotailed = 2 * min(
    np.mean(z_boot >= 0),
    np.mean(z_boot <= 0),
)
print(f"  Bootstrap p (two-tailed, H0: ρ=0): {p_twotailed:.4f}")


In [ ]:
# Infant parcelwise: format group-mean delta_ID and ISC to match combined_delta_isc columns.
# Infant subjects are unmatched (sleep vs aeronaut are different people), so delta_ID is
# computed from group means rather than within-subject differences.
# id_sleep, id_movie, isc_movie come from the pivot/groupby in the cell above —
# parcels are in alphabetical region_name order (pandas default for pivot and groupby).

region_names_inf = sorted(inf_df['region_name'].unique())

inf_parcelwise = pd.DataFrame({
    'subject_id':  'infant_group',
    'region_name': region_names_inf,
    'delta_ID':    id_sleep.mean(axis=0) - id_movie.mean(axis=0),
    'ISC':         isc_movie,
    'age':         inf_df['age_months'].mean(),
    'dataset':     'infant',
})

combined_delta_isc_all = pd.concat(
    [combined_delta_isc, inf_parcelwise],
    ignore_index=True
)
combined_delta_isc_all.to_csv('compiled/results/combined_delta_id_isc_all_datasets.csv', index=False)
print(f"Rows by dataset:\n{combined_delta_isc_all.groupby('dataset').size()}")
combined_delta_isc_all.head()

In [ ]:
# Combine per-subject results (HBN + adult) with the infant group bootstrap point.
# res has columns: subject_id, rho, pval, zscore, age_months, dataset
# infant bootstrap gives: rho_obs, zscore_obs, ci_lo/hi (rho scale), ci_lo_z/hi_z (zscore scale)

mean_infant_age_months = inf_df['age_months'].mean()

# Build a one-row dataframe for the infant group that matches res's columns,
# plus CI columns (NaN for matched subjects, filled for the group point).
infant_row = pd.DataFrame([{
    'subject_id':   'infant_group',
    'rho':          rho_observed,
    'pval':         np.nan,
    'zscore':       z_observed,
    'age_months':   mean_infant_age_months,
    'dataset':      'infant',
    'is_group':     True,
    'rho_ci_lo':    rho_ci_lower ,
    'rho_ci_hi':    rho_ci_upper,
    'zscore_ci_lo': z_ci_lower,
    'zscore_ci_hi': z_ci_upper,
    'age_group':    'infant',
}])

# Add CI columns to res (NaN for individual subjects)
res_combined = res.copy()
res_combined['is_group']     = False
res_combined['rho_ci_lo']    = np.nan
res_combined['rho_ci_hi']    = np.nan
res_combined['zscore_ci_lo'] = np.nan
res_combined['zscore_ci_hi'] = np.nan

df_combined = pd.concat([res_combined, infant_row], ignore_index=True)
df_combined.to_csv('compiled/results/isc_deltaid_rho_developmental.csv', index=False)
print(f"Combined: {len(df_combined)} rows ({df_combined[df_combined['is_group']==False].shape[0]} subjects + 1 infant group point)")
print(df_combined.groupby('dataset')[['rho', 'zscore']].describe())

# ── VISUALIZATION ─────────────────────────────────────────────────────────────

fig = plt.figure(figsize=(14, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.38)

# Panel A: bootstrap distribution
ax_a = fig.add_subplot(gs[0])
ax_a.hist(rho_boot, bins=60, color='#4477AA', alpha=0.75, edgecolor='none')
ax_a.axvline(rho_obs, color='black',   lw=2,   label=f'Observed ρ = {rho_obs:.3f}')
ax_a.axvline(ci_lo,   color='#EE6677', lw=1.5, ls='--', label=f'95% CI [{ci_lo:.3f}, {ci_hi:.3f}]')
ax_a.axvline(ci_hi,   color='#EE6677', lw=1.5, ls='--')
ax_a.axvline(0,       color='gray',    lw=1,   ls=':',  label='ρ = 0')
ax_a.set_xlabel('Bootstrap ρ(ISC, ΔID)', fontsize=11)
ax_a.set_ylabel('Count', fontsize=11)
ax_a.set_title('Infant bootstrap distribution\n(unmatched sleep vs. movie)', fontsize=11)
ax_a.legend(fontsize=8)

# Panel B: group-mean maps scatter (ISC vs ΔID across parcels)
ax_b = fig.add_subplot(gs[1])
delta_obs_full = id_sleep.mean(0) - id_movie.mean(0)
ax_b.scatter(isc_movie, delta_obs_full, alpha=0.5, s=15, c='#4477AA', edgecolors='none')
m, b_lin = np.polyfit(isc_movie, delta_obs_full, 1)
xs = np.linspace(isc_movie.min(), isc_movie.max(), 100)
ax_b.plot(xs, m*xs + b_lin, 'k-', lw=1.5)
ax_b.set_xlabel('ISC (group mean)', fontsize=11)
ax_b.set_ylabel('ΔID sleep–movie (group mean)', fontsize=11)
ax_b.set_title(f'Parcel-wise ISC vs ΔID\nρ = {rho_obs:.3f}', fontsize=11)

# Panel C: developmental trajectory with real per-subject data + infant group
ax_c = fig.add_subplot(gs[2])

dataset_colors = {'HBN': '#AAAAAA', 'adult_restmovie': '#888888', 'infant': '#4477AA'}

subj_df = df_combined[df_combined['is_group'] == False]
for ds, grp in subj_df.groupby('dataset'):
    ax_c.scatter(grp['age_months'], grp['zscore'],
                 c=dataset_colors.get(ds, '#CCCCCC'), alpha=0.4, s=10, zorder=1,
                 label=ds)

inf_row = df_combined[df_combined['is_group'] == True].iloc[0]
ax_c.errorbar(
    inf_row['age_months'], inf_row['zscore'],
    yerr=[[inf_row['zscore'] - inf_row['zscore_ci_lo']], [inf_row['zscore_ci_hi'] - inf_row['zscore']]],
    fmt='o', color='#4477AA', markersize=9, capsize=5, lw=2, zorder=3,
    label=f'Infant group (bootstrap)\nz = {inf_row["zscore"]:.2f} [{inf_row["zscore_ci_lo"]:.2f}, {inf_row["zscore_ci_hi"]:.2f}]'
)
ax_c.axhline(0, color='gray', lw=1, ls='--', alpha=0.7)
ax_c.set_xlabel('Age (months)', fontsize=11)
ax_c.set_ylabel('z(ρ(ISC, ΔID))', fontsize=11)
ax_c.set_title('ISC–ΔID coupling across development', fontsize=11)
ax_c.legend(fontsize=8)

plt.suptitle('Developmental trajectory of ISC–ΔID spatial coupling', fontsize=12, y=1.02)
plt.savefig(f'{PLOT_DIR}/isc_deltaid_developmental_trajectory.pdf', format='pdf', transparent=True, bbox_inches='tight')
plt.show()

In [ ]:
# Bin these into 6 approximately equal-sized groups in a data driven way (not just arbitrary cutoffs)
res['age_group'] = pd.qcut(res['age'], q=8)
sns.barplot(x='age_group', y='zscore', data=res, color='lightgray', edgecolor='k')

In [ ]:
def make_groups(age):
    if age < 8: return 'U08'
    if age < 9.5: return '8-9.5'
    if age < 10.5: return '9.5-10.5'
    if age < 12: return '10.5-12'
    if age < 13.5: return '12-13.5'
    if age < 15: return '13.5-15'
    if age < 17: return '15-17'
    else: return 'O17' 

res['age_group'] = res['age'].apply(make_groups)
order = ['U08', '8-9.5', '9.5-10.5', '10.5-12', '12-13.5', '13.5-15', '15-17', 'O17']
sns.pointplot(x='age_group', y='zscore', data=res, color='lightgray', order=order)
# add N per group to x-axis labels
age_group_counts = res['age_group'].value_counts().reindex(order)
age_group_labels = [f'{age_group}\n(n={count})' for age_group, count in age_group_counts.items()]
plt.xticks(ticks=range(len(age_group_labels)), labels=age_group_labels)

In [ ]:
summary_df = pd.DataFrame(columns=['age_group', 'mean_z', 'ci_lower', 'ci_upper', 'sem', 'std', 'count', 'mean_age_months', 'mean_rho', 'rho_ci_lower', 'rho_ci_upper'])
for ag, grp in res.groupby('age_group'):
    mean = grp['zscore'].mean()
    count = grp['zscore'].count()
    std = grp['zscore'].std()
    age_mean = grp['age_months'].mean()
    sem = std / np.sqrt(count)
    ci_lower = mean - 1.96 * sem
    ci_upper = mean + 1.96 * sem

    mean_rho = grp['rho'].mean()
    rho_sem = grp['rho'].std() / np.sqrt(count)
    rho_ci_lower = mean_rho - 1.96 * rho_sem
    rho_ci_upper = mean_rho + 1.96 * rho_sem

    summary_df.loc[len(summary_df)] = [ag, mean, ci_lower, ci_upper, sem, std, count, age_mean, mean_rho, rho_ci_lower, rho_ci_upper]

# add the infant supergroup as its own row
n_boot = 1000
mean_rho = rho_boot.mean()
rho_sem = rho_boot.std() / np.sqrt(n_boot)
rho_ci_lower = np.percentile(rho_boot, 2.5)
rho_ci_upper = np.percentile(rho_boot, 97.5)


summary_df.loc[len(summary_df)] = ['infant',  z_observed, z_ci_lower, z_ci_upper, z_boot.std(), z_boot.std()/np.sqrt(n_boot), n_boot, mean_infant_age_months, rho_observed, rho_ci_lower, rho_ci_upper]
master_order = ['infant'] + order
summary_df = summary_df.set_index('age_group').reindex(master_order).reset_index()
master_order_age = [summary_df.loc[summary_df['age_group'] == ag, 'mean_age_months'].values[0] for ag in master_order]
summary_df


In [ ]:
summary_df.to_csv('compiled/results/isc_deltaid_developmental_summary.csv', index=False)

In [ ]:
np.save('compiled/results/infant_restmovie_isc_deltaid_z_bootstrap_distribution.npy', z_boot)

### dont need to run above

In [ ]:
# summary_df = pd.read_csv('compiled/results/isc_deltaid_developmental_summary.csv')
# z_boot = np.load('compiled/results/infant_restmovie_isc_deltaid_z_bootstrap_distribution.npy')
# create a merged dataset that combines the bootstrap distribution with the per-subject data, and add a column indicating whether each point is from the bootstrap or a real subject

# load in the average FD for each infant subject
mov_fd = par_df[par_df['dataset']=='InfantRestMovie']['movie_FD'].values
rest_fd = par_df[par_df['dataset']=='InfantRestMovie']['rest_FD'].values
mean_FD = np.mean([mov_fd, rest_fd])
counts = par_df[par_df['dataset']=='InfantRestMovie']['sex'].value_counts()
prop = counts['F'] / counts.sum()

bootstrap_df = pd.DataFrame({
    'age_months': np.random.choice(inf_ages, size=len(z_boot)),  # jittered ages for the bootstrap points
    'zscore': z_boot,
    'source': np.repeat('bootstrap', len(z_boot)),
    'dataset': np.repeat('infant_restmovie', len(z_boot)),
    'subject_id': np.repeat('bootstrap', len(z_boot)),
    'mean_FD': np.repeat(mean_FD, len(z_boot)),
    'movie_FD':np.random.choice(mov_fd, size=len(z_boot)),
    'rest_FD':np.random.choice(rest_fd, size=len(z_boot)),
    'sex': np.random.choice(['M', 'F'], size=len(z_boot), p=[1-prop, prop]),
})

subjects_df = res[['age_months', 'zscore', 'subject_id','dataset']].copy()
subjects_df['source'] = 'subject'
# Log age
info = np.array([get_par_info(s) for s in subjects_df['subject_id'].values])
subjects_df['movie_FD'] = info[:,0].astype(float)
subjects_df['rest_FD'] = info[:,1].astype(float)
subjects_df['sex'] = info[:,2]
subjects_df['mean_FD'] = (subjects_df['movie_FD'] + subjects_df['rest_FD']) / 2
subjects_df['log_age']=np.log(subjects_df['age_months']+1)
summary_df['log_age'] = np.log(summary_df['mean_age_months'] + 1)
combined_df = pd.concat([subjects_df, bootstrap_df], ignore_index=True)
combined_df['log_age'] = np.log(combined_df['age_months'] + 1)

combined_df.head()



In [ ]:
# Mixed effects model: zscore ~ log_age + log_age + sex + mean_FD, with random intercepts for dataset
model_log = smf.mixedlm('zscore ~ log_age + movie_FD + rest_FD + sex', data=subjects_df, groups=subjects_df['dataset']).fit(reml=False)
# add clustered standard errors by dataset
# ols_model = smf.ols('zscore ~ log_age ++ movie_FD + rest_FD + sex', data=subjects_df).fit(cov_type='cluster', cov_kwds={'groups': subjects_df['dataset']})

# Compare the two models using AIC and BIC
print(f"MixedLM AIC: {model_log.aic:.2f}, BIC: {model_log.bic:.2f}")
# # Get the coefficient adn p-value of log_age
coef_log_age = model_log.params['log_age']
pval_log_age = model_log.pvalues['log_age']
print(f"Coefficient for log_age: {coef_log_age:.4f}, p-value: {pval_log_age:.3e}")
model_log.summary()


In [ ]:
summary_df

In [ ]:
# Undo the log scaling to get predicted zscore at the mean infant age
predicted_zscore = model_log.predict(new_data)
# print(f"Predicted z-score at mean infant age ({mean_infant_age_months:.1f} months): {predicted_zscore.values[0]:.4f}")

In [ ]:
predicted_zscore

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Check linearity for each continuous predictor
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Age vs z (partial residual plot is better, but simple scatter works)
axes[0].scatter(subjects_df['log_age'], subjects_df['zscore'], alpha=0.5)
axes[0].set_xlabel('Age')
axes[0].set_ylabel('z')
axes[0].set_title('Age vs z')

# Add smoothed line to see non-linearity
from scipy.ndimage import uniform_filter1d
sorted_idx = np.argsort(subjects_df['log_age'])
axes[0].plot(subjects_df['log_age'].iloc[sorted_idx], 
             uniform_filter1d(subjects_df['zscore'].iloc[sorted_idx], size=100),
             'r-', linewidth=2, label='Smoothed')

# FD vs z
axes[1].scatter(subjects_df['mean_FD'], subjects_df['zscore'], alpha=0.5)
axes[1].set_xlabel('FD')
axes[1].set_ylabel('z')
axes[1].set_title('FD vs z')

plt.tight_layout()
plt.show()

In [ ]:
# Residuals vs Fitted
residuals = ols_model.resid
fitted = ols_model.fittedvalues
# Breusch-Pagan test (from residuals vs fitted)
from scipy import stats

# Simple version: correlation between |residuals| and fitted
abs_resid = np.abs(residuals)
correlation, p_value = stats.pearsonr(fitted, abs_resid)

print(f"Heteroscedasticity test:")
print(f"  Correlation(fitted, |residuals|): {correlation:.3f}")
print(f"  p-value: {p_value:.3f}")
print(f"  {'❌ Violation' if p_value < 0.05 else '✅ OK'}")


plt.figure(figsize=(10, 6))
plt.scatter(fitted, residuals, alpha=0.5)
plt.axhline(0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Fitted Values')
plt.ylabel('Residuals')
plt.title('Residuals vs Fitted (Homoscedasticity Check)')

# Add loess smooth to see pattern
from scipy.signal import savgol_filter
sorted_idx = np.argsort(fitted)
try:
    smooth = savgol_filter(residuals.iloc[sorted_idx], 51, 3)
    plt.plot(fitted.iloc[sorted_idx], smooth, 'b-', linewidth=2, label='Trend')
except:
    pass

plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
from statsmodels.stats.sandwich_covariance import cov_cluster

# Get robust SEs
robust_cov = cov_cluster(ols_model, subjects_df['log_age'])
robust_se = np.sqrt(np.diag(robust_cov))

# Compare to original
print("Sensitivity Analysis: Robust vs. Standard SEs")
print(f"{'Parameter':<20} {'Coef':<10} {'Std SE':<10} {'Robust SE':<10} {'p (std)':<10} {'p (robust)':<10}")
print("-" * 80)

for i, param in enumerate(ols_model.params.index):
    coef = ols_model.params[i]
    std_se = ols_model.bse[i]
    rob_se = robust_se[i]
    
    # Calculate p-values
    from scipy import stats
    t_std = coef / std_se
    t_rob = coef / rob_se
    p_std = 2 * (1 - stats.t.cdf(abs(t_std), ols_model.df_resid))
    p_rob = 2 * (1 - stats.t.cdf(abs(t_rob), ols_model.df_resid))
    
    print(f"{param:<20} {coef:>9.4f} {std_se:>9.4f} {rob_se:>9.4f} {p_std:>9.4f} {p_rob:>9.4f}")

print("\nIf robust and standard p-values are similar → conclusions are robust")

In [ ]:
fig,ax = plt.subplots(figsize=(6, 5))

mean_col, ci_lower_col, ci_upper_col, boot_dist, title, point_label = ['mean_z', 'ci_lower', 'ci_upper', z_boot, 'z-scores from bootstrapping','zscore']

sns.scatterplot(x='age_months', y=point_label, data=combined_df[combined_df['subject_id']!='bootstrap'], ax=ax, hue='dataset', 
                palette=[helper.dataset_colors('adult_restmovie'),helper.dataset_colors('hbn')], 
                s=30, alpha=0.8, zorder=-1) # real subjects

sns.scatterplot(x="age_months", y=point_label, data=combined_df[combined_df['subject_id']=='bootstrap'],
                 ax=ax, color=helper.dataset_colors('infant_restmovie'), s=10, zorder=1, 
                 alpha=0.4, marker='^',label='Infant bootstrap') # bootstrap points

# lower left corner, smaller font

plt.legend(title='dataset', labels=['adult_restmovie', 'hbn', 'infant_restmovie'], loc='lower left')
# add  points with error bars

for idx, row in summary_df.iterrows():
    if idx != 0:
        continue
    ax.errorbar(x=row['mean_age_months'], y=row[mean_col],  yerr=[[row[mean_col] - row[ci_lower_col]], [row[ci_upper_col] - row[mean_col]]], 
        fmt='o', c='k', capsize=2, linewidth=2, zorder=12)

# Add on a model prediction for the mean infant age (with CI from the bootstrap)
infant_age = summary_df.loc[summary_df['age_group'] == 'infant', 'mean_age_months'].values[0]
infant_pred_z = model_log.predict(pd.DataFrame({'log_age': [np.log(infant_age)+1], 'movie_FD': [mov_fd.mean()], 'rest_FD': [rest_fd.mean()], 'sex': ['F']})).values[0]
infant_ci_lower = infant_pred_z - (z_boot.mean() - z_ci_lower)
infant_ci_upper = infant_pred_z + (z_ci_upper - z_boot.mean())
ax.errorbar(x=infant_age+4, y=infant_pred_z, fmt='o', c='magenta', capsize=4, linewidth=2, zorder=15, label='Model prediction at mean infant age')

# Add on the log age regression line from the mixed effects model
age_range = np.linspace(res['age_months'].min(), res['age_months'].max(), 100)
log_age_range = np.log(age_range + 1)
predicted_z = model_log.params['Intercept'] + model_log.params['log_age'] * log_age_range
ax.plot(age_range, predicted_z, color='gray', )

# move legend to bottom left corner and make it smaller
ax.legend(title='Dataset', loc='lower left')

# --- bootstrap CI ---
n_boot = 1000
# boot_preds = np.zeros((n_boot, len(age_range)))

# print("Running bootstrap...")
# for i in range(n_boot):
#     if i % 100 == 0:
#         print(f"  {i}/{n_boot}")
    
#     boot_dfs = []
#     for ds in subjects_df['dataset'].unique():
#         ds_data = subjects_df[subjects_df['dataset'] == ds]
#         boot_dfs.append(resample(ds_data, replace=True, random_state=i))
#     df_boot = pd.concat(boot_dfs).reset_index(drop=True)
#     df_boot['log_age'] = np.log(df_boot['age_months'] + 1)
    
#     try:
#         m_boot = smf.mixedlm(
#             'zscore ~ log_age + movie_FD + rest_FD + sex',
#             data=df_boot,
#             groups=df_boot['dataset']
#         ).fit(reml=False, disp=False)
        
#         # fixed effects only
#         boot_preds[i] = (m_boot.fe_params['Intercept'] + 
#                          m_boot.fe_params['log_age'] * log_age_range)
#     except Exception as e:
#         print(f"Boot {i} failed: {e}")
#         boot_preds[i] = np.nan

# print("Bootstrap complete.")


# --- CI ---
ci_lower = np.nanpercentile(boot_preds, 2.5, axis=0)
ci_upper = np.nanpercentile(boot_preds, 97.5, axis=0)

# # # --- fitted line from original model (fixed effects only) ---
y_pred = (model_log.fe_params['Intercept'] + 
          model_log.fe_params['log_age'] * log_age_range)

# --- plot ---
# Center CI around the fitted line
boot_mean = np.nanmean(boot_preds, axis=0)
ci_lower_centered = y_pred - (boot_mean - ci_lower)
ci_upper_centered = y_pred + (ci_upper - boot_mean)

# Plot with centered CI
ax.plot(age_range, y_pred, color='gray', linewidth=1.5, zorder=3)
ax.fill_between(age_range, 
                ci_lower_centered, ci_upper_centered,
                color='gray', alpha=0.2, zorder=2)





# move the text to the top right corner and make it smaller
ax.text(0.95, 0.25, f'β={coef_log_age:.3f}***', transform=ax.transAxes, ha='right', va='top', fontsize=10)

ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.set_xlabel('Age (months)', fontsize=12)
ax.set_ylabel(f'z-score', fontsize=12)
ax.set_title("ΔID-ISC relationship across development", fontsize=12)
sns.despine()
plt.savefig(f'{PLOT_DIR}/isc_deltaid_developmental_scatter_with_bootstrap_log.pdf', format='pdf', transparent=True, bbox_inches='tight')

# Correspondence between ISC and ID by age

In [ ]:
df1 = pd.read_csv('compiled/results/combined_corrs.csv')
# get sex
par_df = pd.read_csv('compiled/info/combined_participant_info.csv')

def get_sex(subject_id):
    participant_info = par_df[ (par_df['participant_id'] == subject_id)]
    if not participant_info.empty:
        return participant_info.sex.values[0]
    
df1['sex']=df1['participant_id'].apply(get_sex)
df1['FD'].fillna(df1['FD'].mean(), inplace=True)
df1['dataset']=df1['dataset'].replace({'AdultRestMovie': 'adult_restmovie', 'HBN': 'hbn', 'Narratives': 'narratives',
                                       'InfantRestMovie': 'infant_restmovie', 'PartlyCloudy': 'partlycloudy'})

## test three different models: linear, quadratic, and log age effects, and compare AIC/BIC

In [ ]:
df1['log_age'] = np.log(df1['age_months'] + 1)  # +1 to handle any zeros

model_log_ml = smf.mixedlm(
    'zscore ~ log_age + FD + sex',
    data=df1,
    groups=df1['dataset']
).fit(reml=False)

model_lme1_ml = smf.mixedlm(
    'zscore ~ age_months + FD + sex',
    data=df1,
    groups=df1['dataset'],
    
).fit(reml=False)

# Add quadratic term
df1['age_months_sq'] = df1['age_months'] ** 2

# Important: center age first to reduce multicollinearity
# between age and age^2 within datasets, which can cause convergence issues in mixed models.
df1['age_c'] = df1['age_months'] - df1['age_months'].mean()

df1['age_c_sq'] = df1['age_c'] ** 2

model_quad_ml = smf.mixedlm(
    'zscore ~ age_c + age_c_sq + FD + sex',
    data=df1,
    groups=df1['dataset']
).fit(reml=False)


# Compare AIC and BIC
results = pd.DataFrame({
    'model': ['linear', 'quadratic', 'log'],
    'AIC': [model_lme1_ml.aic, model_quad_ml.aic, model_log_ml.aic],
    'BIC': [model_lme1_ml.bic, model_quad_ml.bic, model_log_ml.bic],
    'loglik': [model_lme1_ml.llf, model_quad_ml.llf, model_log_ml.llf]
})
print(results.sort_values('AIC'))

# LRT for linear vs quadratic (nested models)
from scipy import stats
lrt_stat = 2 * (model_quad_ml.llf - model_lme1_ml.llf)
p_lrt = stats.chi2.sf(lrt_stat, df=1)  # 1 extra parameter
print(f"\nLRT linear vs quadratic: χ²(1) = {lrt_stat:.3f}, p = {p_lrt:.4f}")

# Add in delta AIC and delta BIC relative to the best model
best_aic = results['AIC'].min()
best_bic = results['BIC'].min()
results['delta_AIC'] = results['AIC'] - best_aic
results['delta_BIC'] = results['BIC'] - best_bic
print("\nModel comparison with delta AIC/BIC:")
print(results.sort_values('AIC'))  


In [ ]:
lme_fit_summary(model_log_ml)

In [ ]:

fig, ax = plt.subplots(figsize=(6, 5))

color_dict = helper.dataset_colors('all')
hue_order = list(color_dict.keys())
color_palette = [color_dict[i] for i in hue_order]

# --- generate smooth age range ---
age_range = np.linspace(df1['age_months'].min(), df1['age_months'].max(), 300)

# --- build prediction dataframe at mean/reference levels ---
pred_df = pd.DataFrame({
    'age_months': age_range,
    'log_age': np.log(age_range + 1),
    'FD': df1['FD'].mean(),
    'sex': df1['sex'].mode()[0],
    'dataset': df1['dataset'].mode()[0]
})
# --- compute fitted line using fixed effects only ---
def predict_fixed_only(model, pred_df):
    """Predict using fixed effects only, ignoring random effects."""
    X = dmatrix(
        model.model.data.design_info,
        pred_df,
        return_type='matrix'
    )
    fe_names = model.fe_params.index.tolist()
    # subset to match dimensions
    X_df = pd.DataFrame(np.array(X), columns=model.model.data.design_info.column_names)
    X_fe = X_df[fe_names].values
    return X_fe @ model.fe_params.values

# fitted line from original model — fixed effects only
y_pred = predict_fixed_only(model_log_ml, pred_df)

# bootstrap predictions — also fixed effects only
n_boot = 1000
boot_preds = np.zeros((n_boot, len(age_range)))

print("Running bootstrap...")
for i in range(n_boot):
    if i % 100 == 0:
        print(f"  {i}/{n_boot}")
    
    boot_dfs = []
    for ds in df1['dataset'].unique():
        ds_data = df1[df1['dataset'] == ds]
        boot_dfs.append(resample(ds_data, replace=True, random_state=i))
    df_boot = pd.concat(boot_dfs).reset_index(drop=True)
    
    try:
        m_boot = smf.mixedlm(
            'zscore ~ log_age + FD + sex',
            data=df_boot,
            groups=df_boot['dataset']
        ).fit(reml=False, disp=False)
        
        boot_preds[i] = predict_fixed_only(m_boot, pred_df)
    except Exception as e:
        print(f"Boot {i} failed: {e}")
        boot_preds[i] = np.nan

print("Bootstrap complete.")

# CI from bootstrap
ci_lower = np.nanpercentile(boot_preds, 2.5, axis=0)
ci_upper = np.nanpercentile(boot_preds, 97.5, axis=0)
# --- compute fitted line from original model ---
y_pred = model_log_ml.predict(pred_df)

# --- plot log fit line + bootstrap CI band ---
ax.fill_between(age_range, ci_lower, ci_upper,
                color='gray', alpha=0.2, zorder=2)
ax.plot(age_range, y_pred, color='gray', linewidth=1.5, zorder=3)

# --- scatter plot ---
sns.scatterplot(
    x='age_months', y='zscore',
    data=df1,
    hue='dataset',
    hue_order=hue_order,
    palette=color_palette,
    s=30, alpha=0.8,
    ax=ax,
    zorder=1
)

# --- beta annotation ---
beta_val = model_log_ml.fe_params['log_age']
p_val = model_log_ml.pvalues['log_age']
stars = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'n.s.'
ax.text(0.95, 0.76, f'β={beta_val:.3f}{stars}',
        transform=ax.transAxes, ha='right', va='top', fontsize=10)

# --- formatting ---
ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.set_xlabel('Age (months)', fontsize=12)
ax.set_ylabel('z-score', fontsize=12)
ax.set_title('Task ID-ISC relationship across development', fontsize=12)
sns.despine()
plt.tight_layout()

plt.savefig(f'{PLOT_DIR}/isc_id_movie_developmental_trajectory_all_datasets_logmodel.pdf',
            format='pdf', transparent=True, bbox_inches='tight')
plt.show()

## Combine partlycloudy and HBN into one dataset to test age ~ ID relationships 

In [ ]:
# # add movie_FD and sex columns to all dataframes
# par_df = pd.read_csv('compiled/info/combined_participant_info.csv')
# def get_sex(subject_id):
#     participant_info = par_df[ (par_df['participant_id'] == subject_id)]
#     if not participant_info.empty:
#         return participant_info.sex.values[0]

# def get_age(subject_id):
#     participant_info = par_df[ (par_df['participant_id'] == subject_id)]
#     if not participant_info.empty:
#         return participant_info.age_months.values[0]

# def get_fd(subject_id):
#     participant_info = par_df[ (par_df['participant_id'] == subject_id)]
#     if not participant_info.empty:
#         fd_col = f'movie_FD'
#         if fd_col in participant_info.columns:
#             return participant_info[fd_col].values[0]
#         else:
#             return


# pc_df = pd.read_csv('compiled/results/partlycloudy_compiled_results.csv')
# pc_df = pc_df.pivot_table(
#     index=['subject_id', 'region_name', 'task','age_months','dataset'], columns='measure', values='score').reset_index()
# pc_df = pd.read_csv('compiled/results/partlycloudy_compiled_results.csv')
# hbn_df = pd.read_csv('compiled/results/hbn_compiled_results.csv')
# hbn_df = hbn_df[hbn_df['task'] != 'rest'].reset_index(drop=True)
# inf_df = pd.read_csv('compiled/results/infant_restmovie_compiled_results.csv')
# inf_df = inf_df[inf_df['task'] != 'sleep'].reset_index(drop=True)
# nar_df = pd.read_csv('compiled/results/narratives_compiled_results.csv').groupby(['subject_id', 'region_name', 'measure'])['score'].mean().reset_index()
# adu_df = pd.read_csv('compiled/results/adult_restmovie_compiled_results.csv')
# adu_df = adu_df[adu_df['task'] != 'rest'].reset_index(drop=True)
# nar_df['age_months'] = nar_df['subject_id'].apply(get_age)
# nar_df['dataset'] = 'narratives'
# nar_df['task'] = 'narratives'

# pc_df = pc_df.pivot_table(
#     index=['subject_id', 'region_name', 'task','age_months','dataset'], columns='measure', values='score').reset_index()
# hbn_df = hbn_df.pivot_table(
#     index=['subject_id', 'region_name', 'task','age_months','dataset'], columns='measure', values='score').reset_index()
# inf_df = inf_df.pivot_table(
#     index=['subject_id', 'region_name', 'task','age_months','dataset'], columns='measure', values='score').reset_index()
# nar_df = nar_df.pivot_table(
#     index=['subject_id', 'region_name','task', 'age_months','dataset'], columns='measure', values='score').reset_index()
# adu_df = adu_df.pivot_table(
#     index=['subject_id', 'region_name', 'task','age_months','dataset'], columns='measure', values='score').reset_index()

# pc_df['sex'] = pc_df['subject_id'].apply(get_sex)
# pc_df['movie_FD'] = pc_df['subject_id'].apply(get_fd)
# hbn_df['sex'] = hbn_df['subject_id'].apply(get_sex)
# hbn_df['movie_FD'] = hbn_df['subject_id'].apply(get_fd)
# nar_df['movie_FD'] = nar_df['subject_id'].apply(get_fd)
# nar_df['sex'] = nar_df['subject_id'].apply(get_sex)
# adu_df['movie_FD'] = adu_df['subject_id'].apply(get_fd)
# adu_df['sex'] = adu_df['subject_id'].apply(get_sex)
# inf_df['movie_FD'] = inf_df['subject_id'].apply(get_fd)
# inf_df['sex'] = inf_df['subject_id'].apply(get_sex)

# combined_df= pd.concat([pc_df, hbn_df, nar_df, adu_df, inf_df], ignore_index=True)

In [ ]:
combined_df.to_csv('compiled/results/combined_ID_ISC_with_covariates.csv', index=False)

In [ ]:
combined_df = pd.read_csv('compiled/results/combined_ID_ISC_with_covariates.csv')

In [ ]:
REGION_ORDER = get_region_order()
combined_df['dataset'] = [str(s) for s in combined_df['dataset'].values]
combined_df['dataset'] = combined_df['dataset'].astype('category')
combined_df['sex'] = combined_df['sex'].astype('category')
combined_df['log_age'] = np.log(combined_df['age_months'] + 1)
combined_df = combined_df.reset_index(drop=True)

# drop NAs
combined_df = combined_df.dropna(subset=['TPHATE_DiffOp_IDE', 'ISC', 'log_age', 'movie_FD', 'sex'])

formula = "TPHATE_DiffOp_IDE ~ ISC + log_age + movie_FD + sex + (1|dataset)"
a = stats_helpers.parcelwise_regression(combined_df, "log_age", yname="TPHATE_DiffOp_IDE",
                                             formula=formula, region_order=REGION_ORDER,
                                             alpha=0.05, fdr_method='fdr_bh',lme=True)

a = a.set_index('region_name').reindex(REGION_ORDER).reset_index()

for var, label in [('log_age', 'log_age'), ('ISC', 'isc')]:
    sig_mask = a[f'sig_{var}_fdr'].values
    coef_masked = a[f'coef_{var}'].values * sig_mask
    print(f"Var: {var}, surviving significant coefs: {np.sum(sig_mask)}; negative: {np.sum(coef_masked < 0)}, positive: {np.sum(coef_masked > 0)}")
    cbar_range, this_cmap = helper.determine_colorbar_range(coef_masked, helper.diverging_colormap_gpu(), symmetric=True)
    helper.generate_surface_plot(
        coef_masked,
        image_fn=f'{PLOT_DIR}/all_datasets_no_inter_log_age_predicts_ide_{label}_coef_surface.pdf',
        atlas='Schaefer', cmap=this_cmap, cbar_range=cbar_range,
        surf_type='fslr', target_density='32k', method='linear',
        include_cbar=True, title=rf'$\beta$_{var}- combined', threshold=None, mask_medial_wall=True
    )

In [ ]:
# Print out the networks that show significant log_age effects
sig_regions = a[a['sig_log_age_fdr'] == 1]['region_name'].values
directions = [-1 if s < 0 else 1 for s in a[a['sig_log_age_fdr'] == 1]['coef_log_age'].values]
networks = {}
for region, direction in zip(sig_regions, directions):
    network = helper.get_schaefer_network(region)
    networks[network] = networks.get(network, 0) + 1*direction   
print("Networks with significant log_age effects:")
# Sort networks by count
networks = dict(sorted(networks.items(), key=lambda item: item[1], reverse=True))

print(networks)

In [ ]:
pc_df = pd.read_csv('compiled/results/partlycloudy_compiled_results.csv')

pc_df = pc_df.pivot_table(
    index=['subject_id', 'region_name', 'task','age_months','dataset'], columns='measure', values='score').reset_index()


In [ ]:
pc_df = pd.read_csv('compiled/results/partlycloudy_compiled_results.csv')
hbn_df = pd.read_csv('compiled/results/hbn_compiled_results.csv')

hbn_df = hbn_df[hbn_df['task'] != 'rest'].reset_index(drop=True)
pc_df = pc_df.pivot_table(
    index=['subject_id', 'region_name', 'task','age_months','dataset'], columns='measure', values='score').reset_index()
hbn_df = hbn_df.pivot_table(
    index=['subject_id', 'region_name', 'task','age_months','dataset'], columns='measure', values='score').reset_index()

# add movie_FD and sex columns to both dataframes
par_df = pd.read_csv('compiled/info/combined_participant_info.csv')
def get_sex(subject_id):
    participant_info = par_df[ (par_df['participant_id'] == subject_id)]
    if not participant_info.empty:
        return participant_info.sex.values[0]
def get_fd(subject_id):
    participant_info = par_df[ (par_df['participant_id'] == subject_id)]
    if not participant_info.empty:
        fd_col = f'movie_FD'
        if fd_col in participant_info.columns:
            return participant_info[fd_col].values[0]
        else:
            return

pc_df['sex'] = pc_df['subject_id'].apply(get_sex)
pc_df['movie_FD'] = pc_df['subject_id'].apply(get_fd)
hbn_df['sex'] = hbn_df['subject_id'].apply(get_sex)
hbn_df['movie_FD'] = hbn_df['subject_id'].apply(get_fd)

combined_df= pd.concat([pc_df, hbn_df], ignore_index=True)

In [ ]:
combined_df['log_age']= np.log(combined_df['age_months'] + 1)
REGION_ORDER = get_region_order()   
formula = "TPHATE_DiffOp_IDE ~ ISC + log_age + movie_FD + sex + (1|dataset)"
a = stats_helpers.parcelwise_regression(combined_df, "log_age", yname="TPHATE_DiffOp_IDE",
                                             formula=formula, region_order=REGION_ORDER,
                                             alpha=0.05, fdr_method='fdr_bh',lme=True)
a = a.set_index('region_name').reindex(REGION_ORDER).reset_index()

for var, label in [('log_age', 'log_age'), ('ISC', 'isc')]:
    sig_mask = a[f'sig_{var}_fdr'].values
    coef_masked = a[f'coef_{var}'].values * sig_mask
    cbar_range, this_cmap = helper.determine_colorbar_range(coef_masked, helper.diverging_colormap_gpu(), symmetric=True)
    helper.generate_surface_plot(
        coef_masked,
        image_fn=f'{PLOT_DIR}/HBN_PC_no_inter_log_age_predicts_ide_{label}_coef_surface.pdf',
        atlas='Schaefer', cmap=this_cmap, cbar_range=cbar_range,
        surf_type='fslr', target_density='32k', method='linear',
        include_cbar=True, title=rf'$\beta$_{var} HBN + PC log', threshold=None, mask_medial_wall=True
    )

In [ ]:
# REGION_ORDER = get_region_order()   
# formula = "TPHATE_DiffOp_IDE ~ ISC * age_months + movie_FD + sex"
# a = stats_helpers.parcelwise_regression(pc_df, "age_months", yname="TPHATE_DiffOp_IDE",
#                                              formula=formula, region_order=REGION_ORDER,
#                                              alpha=0.05, fdr_method='fdr_bh',lme=False)
# a = a.set_index('region_name').reindex(REGION_ORDER).reset_index()

# for var, label in [('age_months', 'age_months'), ('ISC', 'isc'), ('ISC_age_months', 'isc_age_interaction')]:
#     sig_mask = a[f'sig_{var}_fdr'].values
#     coef_masked = a[f'coef_{var}'].values * sig_mask
#     cbar_range, this_cmap = helper.determine_colorbar_range(coef_masked, helper.diverging_colormap_gpu(), symmetric=True)
#     helper.generate_surface_plot(
#         coef_masked,
#         image_fn=f'{PLOT_DIR}/pc_w_inter_age_predicts_ide_{label}_coef_surface.pdf',
#         atlas='Schaefer', cmap=this_cmap, cbar_range=cbar_range,
#         surf_type='fslr', target_density='32k', method='linear',
#         include_cbar=True, title=rf'$\beta$_{var} PC w inter', threshold=None, mask_medial_wall=True
#     )

# REGION_ORDER = get_region_order()   
pc_df['log_age'] = np.log(pc_df['age_months'] + 1)
formula = "TPHATE_DiffOp_IDE ~ ISC + log_age + movie_FD + sex"
a = stats_helpers.parcelwise_regression(pc_df, "log_age", yname="TPHATE_DiffOp_IDE",
                                             formula=formula, region_order=REGION_ORDER,
                                             alpha=0.05, fdr_method='fdr_bh',lme=False)
a = a.set_index('region_name').reindex(REGION_ORDER).reset_index()

for var, label in [('log_age', 'log_age'), ('ISC', 'isc')]:
    sig_mask = a[f'sig_{var}_fdr'].values
    coef_masked = a[f'coef_{var}'].values * sig_mask
    print(f"Var: {var}, surviving significant coefs: {np.sum(sig_mask)}")
    cbar_range, this_cmap = helper.determine_colorbar_range(coef_masked, helper.diverging_colormap_gpu(), symmetric=True)
    helper.generate_surface_plot(
        coef_masked,
        image_fn=f'{PLOT_DIR}/PartlyCloudy_no_inter_log_age_predicts_ide_{label}_coef_surface.pdf',
        atlas='Schaefer', cmap=this_cmap, cbar_range=cbar_range,
        surf_type='fslr', target_density='32k', method='linear',
        include_cbar=True, title=rf'$\beta$_{var} Partly Cloudy', threshold=None, mask_medial_wall=True
    )

# REGION_ORDER = get_region_order()   
# formula = "TPHATE_DiffOp_IDE ~ ISC * age_months + movie_FD + sex"
# a = stats_helpers.parcelwise_regression(hbn_df, "age_months", yname="TPHATE_DiffOp_IDE",
#                                              formula=formula, region_order=REGION_ORDER,
#                                              alpha=0.05, fdr_method='fdr_bh',lme=False)
# a = a.set_index('region_name').reindex(REGION_ORDER).reset_index()

# for var, label in [('age_months', 'age_months'), ('ISC', 'isc'), ('ISC_age_months', 'isc_age_interaction')]:
#     sig_mask = a[f'sig_{var}_fdr'].values
#     coef_masked = a[f'coef_{var}'].values * sig_mask
#     cbar_range, this_cmap = helper.determine_colorbar_range(coef_masked, helper.diverging_colormap_gpu(), symmetric=True)
#     helper.generate_surface_plot(
#         coef_masked,
#         image_fn=f'{PLOT_DIR}/hbn_w_inter_age_predicts_ide_{label}_coef_surface.pdf',
#         atlas='Schaefer', cmap=this_cmap, cbar_range=cbar_range,
#         surf_type='fslr', target_density='32k', method='linear',
#         include_cbar=True, title=rf'$\beta$_{var} HBN w inter', threshold=None, mask_medial_wall=True
#     )

# REGION_ORDER = get_region_order()   
formula = "TPHATE_DiffOp_IDE ~ ISC + log_age + movie_FD + sex"
hbn_df['log_age'] = np.log(hbn_df['log_age'] + 1)
a = stats_helpers.parcelwise_regression(hbn_df, "log_age", yname="TPHATE_DiffOp_IDE",
                                             formula=formula, region_order=REGION_ORDER,
                                             alpha=0.05, fdr_method='fdr_bh',lme=False)
a = a.set_index('region_name').reindex(REGION_ORDER).reset_index()

for var, label in [('log_age', 'log_age'), ('ISC', 'isc')]:
    sig_mask = a[f'sig_{var}_fdr'].values
    coef_masked = a[f'coef_{var}'].values * sig_mask
    print(f"Var: {var}, surviving significant coefs: {np.sum(sig_mask)}")
    cbar_range, this_cmap = helper.determine_colorbar_range(coef_masked, helper.diverging_colormap_gpu(), symmetric=True)
    helper.generate_surface_plot(
        coef_masked,
        image_fn=f'{PLOT_DIR}/HBN_no_inter_log_age_predicts_ide_{label}_coef_surface.pdf',
        atlas='Schaefer', cmap=this_cmap, cbar_range=cbar_range,
        surf_type='fslr', target_density='32k', method='linear',
        include_cbar=True, title=rf'$\beta$_{var} HBN', threshold=None, mask_medial_wall=True
    )

# Combining across datasets to look at change in average ID as an effect of age

In [ ]:
# Load datasets separate into ID and ISC
combined_df = combined_df[combined_df['task'] != 'mickey'].reset_index(drop=True)
# set dataset as categorical
combined_df['dataset'] = combined_df['dataset'].astype('category')
combined_df['sex']=combined_df['sex'].astype('category')
combined_df['region_name']=combined_df['region_name'].astype('category')


In [ ]:
lme_isc_combined1 = smf.mixedlm('ISC ~ age_months + movie_FD + sex', data=mean_df, groups=mean_df['dataset']).fit()
lme_ide_combined1 = smf.mixedlm('TPHATE_DiffOp_IDE ~ age_months + movie_FD + sex', data=mean_df, groups=mean_df['dataset']).fit()
print("Combined ISC regression results:")
print(lme_isc_combined1.summary())
print("\nCombined IDE regression results:")
print(lme_ide_combined1.summary())

In [ ]:
mean_df = combined_df.groupby(['subject_id', 'dataset', 'age_months', 'movie_FD', 'sex']).agg({'ISC': 'mean', 'TPHATE_DiffOp_IDE':'mean'}).reset_index()
mean_df['log_age'] = np.log(mean_df['age_months'] + 1)


lme_isc_combined = smf.mixedlm('ISC ~ log_age + movie_FD + sex', data=mean_df, groups=mean_df['dataset']).fit()
lme_ide_combined = smf.mixedlm('TPHATE_DiffOp_IDE ~ log_age + movie_FD + sex', data=mean_df, groups=mean_df['dataset']).fit()
print("Combined ISC regression results:")
print(lme_isc_combined.summary())
print("\nCombined IDE regression results:")
print(lme_ide_combined.summary())

color_dict = helper.dataset_colors('all')
hue_order = list(color_dict.keys())
color_palette = [color_dict[i] for i in hue_order]

# CONFIDENCE INTERVALS FOR THE REGRESSION LINE
# --- generate smooth age range ---
age_range = np.linspace(mean_df['age_months'].min(), mean_df['age_months'].max(), 300)

# --- compute fitted line using fixed effects only ---
def predict_fixed_only(model, pred_df):
    """Predict using fixed effects only, ignoring random effects."""
    X = dmatrix(
        model.model.data.design_info,
        pred_df,
        return_type='matrix'
    )
    fe_names = model.fe_params.index.tolist()
    # subset to match dimensions
    X_df = pd.DataFrame(np.array(X), columns=model.model.data.design_info.column_names)
    X_fe = X_df[fe_names].values
    return X_fe @ model.fe_params.values

def bootstrap_predictions(orig_model, dataframe, formula, pred_df, n_boot=1000):
    """Bootstrap predictions using fixed effects only."""
    boot_preds = np.zeros((n_boot, len(pred_df)))
    for i in range(n_boot):
        if i % 100 == 0:
            print(f"  {i}/{n_boot}")
        
        boot_dfs = []
        for ds in dataframe['dataset'].unique():
            ds_data = dataframe[dataframe['dataset'] == ds]
            boot_dfs.append(resample(ds_data, replace=True, random_state=i))
        df_boot = pd.concat(boot_dfs).reset_index(drop=True)
        
        # try:
        m_boot = smf.mixedlm(
            formula,
            data=df_boot,
            groups=df_boot['dataset']
        ).fit(reml=False, disp=False)
        boot_preds[i] = predict_fixed_only(m_boot, pred_df)
        # except Exception as e:
        #     print(f"Boot {i} failed: {e}")
        #     boot_preds[i] = np.nan
    # CI from bootstrap
    ci_lower = np.nanpercentile(boot_preds, 2.5, axis=0)
    ci_upper = np.nanpercentile(boot_preds, 97.5, axis=0)
    # --- compute fitted line from original model ---
    y_pred = orig_model.predict(pred_df)
    return y_pred, ci_lower, ci_upper
        
# --- build prediction dataframe at mean/reference levels ---
pred_df_isc = pd.DataFrame({
    'age_months': age_range,
    'log_age': np.log(age_range + 1),
    'movie_FD': mean_df['movie_FD'].mean(),
    'sex': mean_df['sex'].mode()[0],
    'dataset': mean_df['dataset'].mode()[0]
})

# --- build prediction dataframe at mean/reference levels ---
pred_df_ide = pd.DataFrame({
    'age_months': age_range,
    'log_age': np.log(age_range + 1),
    'movie_FD': mean_df['movie_FD'].mean(),
    'sex': mean_df['sex'].mode()[0],
    'dataset': mean_df['dataset'].mode()[0]
})
y_pred_isc, ci_lower_isc, ci_upper_isc = bootstrap_predictions(lme_isc_combined, mean_df, 'ISC ~ log_age + movie_FD + C(sex)', pred_df_isc, n_boot=10000)
print("Bootstrap complete ISC")
y_pred_ide, ci_lower_ide, ci_upper_ide = bootstrap_predictions(lme_ide_combined, mean_df, 'TPHATE_DiffOp_IDE ~ log_age + movie_FD + C(sex)', pred_df_ide, n_boot=10000)
print("Bootstrap complete IDE")


In [ ]:
fig, axes = plt.subplots(1,2, figsize=(10, 4))
plt.rcParams['font.family'] = 'Arial'

ax=axes[0]
sns.scatterplot(
    x='age_months', y='ISC',
    data=mean_df,
    hue='dataset',
    hue_order=hue_order,
    palette=color_palette,
    s=30, alpha=0.8,
    ax=ax,
    zorder=1
)

# --- plot log fit line + bootstrap CI band ---
ax.fill_between(age_range, ci_lower_isc, ci_upper_isc,
                color='gray', alpha=0.2, zorder=2)
ax.plot(age_range, y_pred_isc, color='gray', linewidth=1.5, zorder=3)
beta_age_isc_combined = lme_isc_combined.params['log_age']
p_age_isc_combined = lme_isc_combined.pvalues['log_age']
sig_isc = helper.get_asterisks_pvalue(p_age_isc_combined)
ax.text(0.92, 0.84, f'β={beta_age_isc_combined:.3f}{sig_isc}', 
    transform=ax.transAxes, ha='right', va='top', fontsize=12)
ax.set_xlabel('Age (months)', fontsize=12); ax.set_ylabel('ISC', fontsize=12); ax.set_title('Average ISC predicted by age')


ax=axes[1]
sns.scatterplot(
    x='age_months', y='TPHATE_DiffOp_IDE',
    data=mean_df,
    hue='dataset',
    hue_order=hue_order,
    palette=color_palette,
    s=30, alpha=0.8,
    ax=ax,
    zorder=1
)

# --- plot log fit line + bootstrap CI band ---
ax.fill_between(age_range, ci_lower_ide, ci_upper_ide,
                color='gray', alpha=0.2, zorder=2)
ax.plot(age_range, y_pred_ide, color='gray', linewidth=1.5, zorder=3)
# --- plot log fit line + bootstrap CI band ---
ax.fill_between(age_range, ci_lower_ide, ci_upper_ide,
                color='gray', alpha=0.2, zorder=2)
ax.plot(age_range, y_pred_ide, color='gray', linewidth=1.5, zorder=3)
beta_age_ide_combined = lme_ide_combined.params['log_age']
p_age_ide_combined = lme_ide_combined.pvalues['log_age']
sig_isc = helper.get_asterisks_pvalue(p_age_ide_combined)
ax.text(0.92, 0.44, f'β={beta_age_ide_combined:.3f}{sig_isc}', 
    transform=ax.transAxes, ha='right', va='top', fontsize=12)
ax.set_xlabel('Age (months)', fontsize=12); ax.set_ylabel('Dimensionality', fontsize=12); ax.set_title('Average IDE predicted by age')
sns.despine(); plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/all_datasets_combined_age_effects_isc_ide.pdf', format='pdf', transparent=True)

In [ ]:
!open $PLOT_DIR

In [ ]:
# get all values for log_age coefficient 
lme_isc_combined.params['log_age'], lme_isc_combined.pvalues['log_age'],

In [ ]:
lme_ide_combined.params['log_age'], lme_ide_combined.pvalues['log_age'] 
lme_ide_combined.summary()

In [ ]:
# THIS IS ONLY FOR HBN/PC

import matplotlib.pyplot as plt

# Set Arial as the default font
plt.rcParams['font.family'] = 'Arial'
# par_df_hbn = par_df_hbn.reset_index()
# Pool together the partly cloudy and healthy brain network datasets to run a mega-analysis of age effects on ISC and IDE across the full developmental span,
# controlling for dataset, mean FD, and sex
# First, we need to add a 'dataset' column to each dataframe and then concatenate them
single_isc_pc = single_isc.copy()
single_isc_pc['dataset'] = 'PartlyCloudy'
single_ide_pc = single_ide.copy()
single_ide_pc['dataset'] = 'PartlyCloudy'
single_isc_hbn_copy = single_isc_hbn.copy()
single_isc_hbn_copy['dataset'] = 'HBN'
single_ide_hbn_copy = single_ide_hbn.copy()
single_ide_hbn_copy['dataset'] = 'HBN'

single_isc_pc['sex'] = single_isc_pc['subject_id'].map(par_df_pc.set_index('participant_id')['sex'])
single_ide_pc['sex'] = single_ide_pc['subject_id'].map(par_df_pc.set_index('participant_id')['sex'])
single_ide_hbn_copy['sex'] = single_ide_hbn_copy['subject_id'].map(par_df_hbn.set_index('subject_id')['sex'])
single_isc_hbn_copy['sex'] = single_isc_hbn_copy['subject_id'].map(par_df_hbn.set_index('subject_id')['sex'])


single_isc_combined = pd.concat([single_isc_pc, single_isc_hbn_copy], ignore_index=True)
single_ide_combined = pd.concat([single_ide_pc, single_ide_hbn_copy], ignore_index=True)
single_isc_combined['age_y'] = single_isc_combined['age_months'] / 12
single_ide_combined['age_y'] = single_ide_combined['age_months'] / 12

# Now we can run regression analyses to predict ISC and IDE from age, controlling for dataset, mean FD, and sex
# Use a LME model with random intercepts for dataset
lme_isc_combined = smf.mixedlm('score ~ age_y + mean_FD + sex', data=single_isc_combined, groups=single_isc_combined['dataset']).fit()
lme_ide_combined = smf.mixedlm('score ~ age_y + mean_FD + sex', data=single_ide_combined, groups=single_ide_combined['dataset']).fit()
print("Combined ISC regression results:")
print(lme_isc_combined.summary())
print("\nCombined IDE regression results:")
print(lme_ide_combined.summary())

# Plot the combined data with regression lines and betas and p-values for the age predictor
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Create color palette for datasets
dataset_palette = {
    'PartlyCloudy': helper.dataset_colors('partlycloudy'),
    'HBN': helper.dataset_colors('hbn')
}

# ISC plot
ax = axes[0]
sns.scatterplot(data=single_isc_combined, x='age_y', y='score', hue='dataset', 
                palette=dataset_palette, s=24, ax=ax, legend=True, edgecolor='k', linewidth=0.1)

sns.regplot(data=single_isc_combined, x='age_y', y='score', scatter=False, ax=ax, color='gray',line_kws={'linewidth': 2})
beta_age_isc_combined = lme_isc_combined.params['age_y']
p_age_isc_combined = lme_isc_combined.pvalues['age_y']
sig_isc = helper.get_asterisks_pvalue(p_age_isc_combined)
# Compute rho value for scatter of ISC vs agemonths
rho_isc, p_rho_isc = stats.spearmanr(single_isc_combined['age_y'], single_isc_combined['score'])
ax.text(0.92, 0.64, f'β={beta_age_isc_combined:.3f}{sig_isc}\nρ={rho_isc:.2f}', 
    transform=ax.transAxes, ha='right', va='top', fontsize=12)
ax.set_xlabel('Age (years)', fontsize=12); ax.set_ylabel('ISC', fontsize=12); ax.set_title('Average ISC ~ age + FD + (1|dataset)')

# IDE plot
ax = axes[1]
sns.scatterplot(data=single_ide_combined, x='age_y', y='score', hue='dataset',  
                palette=dataset_palette, s=24,  ax=ax, legend=True, edgecolor='k', linewidth=0.1)
sns.regplot(data=single_ide_combined, x='age_y', y='score', scatter=False, ax=ax, color='gray', line_kws={'linewidth': 2})
beta_age_ide_combined = lme_ide_combined.params['age_y']
p_age_ide_combined = lme_ide_combined.pvalues['age_y']
rho_ide, p_rho_ide = stats.spearmanr(single_ide_combined['age_y'], single_ide_combined['score'])
sig_ide = helper.get_asterisks_pvalue(p_age_ide_combined)
ax.text(0.98, 0.4, f'β={beta_age_ide_combined:.3f}{sig_ide}\nρ={rho_ide:.2f}', 
    transform=ax.transAxes, ha='right', va='top',  fontsize=12)
ax.set_xlabel('Age (years)', fontsize=12); ax.set_ylabel('Dimensionality', fontsize=12); ax.set_title('Average ID ~ age + FD + (1|dataset)')

sns.despine(); plt.tight_layout()
#plt.savefig(f'{PLOT_DIR}/combined_age_effects_isc_ide.pdf', format='pdf', transparent=True)
#plt.show()